In [ ]:
# STEP 1 — Import Libraries
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ.setdefault("OMP_NUM_THREADS", "4")

import sys, re, math, random, json, time, zlib, glob, shutil, hashlib, subprocess, importlib
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from skimage.feature import graycomatrix, graycoprops
from skimage.metrics import structural_similarity as ssim_metric
import gymnasium as gym
from gymnasium import spaces
import torch
import stable_baselines3
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED); torch.manual_seed(GLOBAL_SEED)
print("Libraries imported successfully.")
print("Python:", sys.version.split()[0], "| NumPy:", np.__version__, "| OpenCV:", cv2.__version__, "| Torch:", torch.__version__)
print("Gymnasium:", gym.__version__, "| Stable-Baselines3:", stable_baselines3.__version__, "| MaskablePPO available")

In [ ]:
# STEP 2 — Configuration, hyperparameters and the independent output directory (v3 experiment)
ROOT = "Final_PPO_Implementation_v3"
OLD_ROOT = "Final_PPO_Implementation_v2"         # v2 experiment: READ ONLY
V1_ROOT = "Final_PPO_Implementation"             # v1 experiment: READ ONLY
PROCESSED_DIR        = os.path.join(ROOT, "processed_data")
PROCESSED_COVER_DIR  = os.path.join(PROCESSED_DIR, "covers")
PROCESSED_SECRET_DIR = os.path.join(PROCESSED_DIR, "secrets")
MODEL_DIR            = os.path.join(ROOT, "models")
CHECKPOINT_DIR       = os.path.join(ROOT, "checkpoints")
LOG_DIR              = os.path.join(ROOT, "logs")
STEGO_DIR            = os.path.join(ROOT, "stego_images")
META_DIR             = os.path.join(ROOT, "metadata")
RECOVERED_DIR        = os.path.join(ROOT, "recovered_images")
TABLE_DIR            = os.path.join(ROOT, "tables")
PLOT_DIR             = os.path.join(ROOT, "graphs")
NOISE_DIR            = os.path.join(ROOT, "noise_experiments")
NOISY_STEGO_DIR      = os.path.join(NOISE_DIR, "noisy_stego_images")
NOISY_RECOVERED_DIR  = os.path.join(NOISE_DIR, "noisy_recovered_secrets")
NOISE_RESULT_DIR     = os.path.join(NOISE_DIR, "results")
CACHE_DIR            = os.path.join(ROOT, "cache")
COMPARISON_DIR       = os.path.join(ROOT, "comparison_results")
PER_SAMPLE_DIR       = os.path.join(COMPARISON_DIR, "per_sample")
SAMPLE_DIR           = os.path.join(ROOT, "sample_outputs")
RECEIVER_FILE        = os.path.join(ROOT, "receiver_final_ppo_v2.py")      # receiver code is unchanged, so the module name is unchanged
PAIR_FILE            = os.path.join(PROCESSED_DIR, "pair_assignments_v3.csv")

# v2 files that are only READ (pair table, feature cache, processed images, old model for the compatibility check)
OLD_PAIR_FILE = os.path.join(OLD_ROOT, "processed_data", "pair_assignments_v2.csv")
OLD_CACHE_DIR = os.path.join(OLD_ROOT, "cache")
OLD_MODEL_ZIP = os.path.join(OLD_ROOT, "models", "ppo_dna_ppo_final_v2_best.zip")

EXISTING_PATHS = ["Improved_DNA_Steganography", V1_ROOT, OLD_ROOT, "outputs", "models", "checkpoints", "receiver_standalone.py"]
_new_root = os.path.abspath(ROOT)
for p in EXISTING_PATHS:
    ap = os.path.abspath(p)
    assert _new_root != ap and not _new_root.startswith(ap + os.sep) and not ap.startswith(_new_root + os.sep), \
        f"The new output directory must be separate from the existing location: {p}"

def snapshot_files(paths):
    snap = {}
    for p in paths:
        if os.path.isfile(p):
            st = os.stat(p); snap[os.path.abspath(p)] = (st.st_size, st.st_mtime_ns)
        for dp, _, fs in os.walk(p):
            for f in fs:
                fp = os.path.join(dp, f); st = os.stat(fp)
                snap[os.path.abspath(fp)] = (st.st_size, st.st_mtime_ns)
    return snap

def assert_in_root(path):
    assert os.path.abspath(path).startswith(_new_root + os.sep), f"Path is outside the new output directory: {path}"

EXISTING_SNAPSHOT = snapshot_files(EXISTING_PATHS)
for d in [PROCESSED_COVER_DIR, PROCESSED_SECRET_DIR, MODEL_DIR, CHECKPOINT_DIR, LOG_DIR, STEGO_DIR, META_DIR,
          RECOVERED_DIR, TABLE_DIR, PLOT_DIR, NOISY_STEGO_DIR, NOISY_RECOVERED_DIR, NOISE_RESULT_DIR,
          CACHE_DIR, PER_SAMPLE_DIR, SAMPLE_DIR]:
    os.makedirs(d, exist_ok=True)

# Image and block parameters (unchanged)
COVER_SIZE = 512
SECRET_SIZES = [64, 128, 256, 512]
SECRET_SIZE = 128
BLOCK_SIZE = 8
NUM_BLOCKS_PER_SIDE = COVER_SIZE // BLOCK_SIZE
NUM_BLOCKS = NUM_BLOCKS_PER_SIDE ** 2
NUM_FEATURES = 4
NUM_DNA_RULES = 8
NUM_LSB_OPTIONS = 3
NUM_PIXEL_LOCATIONS = BLOCK_SIZE * BLOCK_SIZE
NUM_EMBED_FLAGS = 2
MAX_LSB = NUM_LSB_OPTIONS
EMBED_SPAN = 64
MAX_BLOCK_BITS = EMBED_SPAN * 3 * MAX_LSB
FEATURE_LSB_MASK = (0xFF << MAX_LSB) & 0xFF
THEORETICAL_MAX_CAPACITY_BITS = COVER_SIZE * COVER_SIZE * 3 * MAX_LSB
assert NUM_BLOCKS == 4096 and THEORETICAL_MAX_CAPACITY_BITS == NUM_BLOCKS * MAX_BLOCK_BITS

# Legacy (v2) environment layout: used only by STEP 21-23B to reproduce / audit v2
HEAD_SIZES = [NUM_BLOCKS, NUM_DNA_RULES, NUM_LSB_OPTIONS, NUM_PIXEL_LOCATIONS, NUM_EMBED_FLAGS]
HEAD_OFFSETS = np.concatenate([[0], np.cumsum(HEAD_SIZES)])
NUM_EXTRA_OBS = 3
OBS_DIM = NUM_BLOCKS * NUM_FEATURES + NUM_BLOCKS + NUM_EXTRA_OBS
# Sequential (v3) environment layout: the agent sees the CURRENT block and decides [DNA rule, LSB depth, location, embed/skip]
SEQ_HEAD_SIZES = [NUM_DNA_RULES, NUM_LSB_OPTIONS, NUM_PIXEL_LOCATIONS, NUM_EMBED_FLAGS]
SEQ_HEAD_OFFSETS = np.concatenate([[0], np.cumsum(SEQ_HEAD_SIZES)])
SEQ_OBS_DIM = 12        # 4 current-block features + 4 image-mean features + progress, capacity pressure, payload ratio, scan position
METADATA_VERSION = "1.0"
MODEL_VERSION = "dna_ppo_seq_v3"

# Data split (identical to v2; pair assignment is verified against v2 in STEP 5B)
TEST_FRACTION = 0.20
VAL_FRACTION = 0.10

# Legacy v2 reward constants (used only by the legacy environment)
PROGRESS_WEIGHT, QUALITY_WEIGHT, DISTORTION_WEIGHT, LSB_PENALTY_WEIGHT = 1.0, 1.0, 0.5, 0.25
SKIP_PENALTY, INVALID_PICK_PENALTY = 0.0002, 0.1
PSNR_FLOOR_DB, PSNR_CEIL_DB = 30.0, 60.0
COMPLETION_BONUS, INCOMPLETE_PENALTY_SCALE, STEP_REWARD_CLIP = 1.0, 5.0, 2.0
USE_FEASIBILITY_MASK = True
MAX_EPISODE_STEPS = NUM_BLOCKS + 500
# Reward v3: R = -sum_t [ SE_t / SE_ref + SSIM_COST_WEIGHT * (1 - SSIM_t) / L_ref ]  (references = plain sequential LSB)
SSIM_COST_WEIGHT = 1.0

# PPO (new model from scratch: the v2 model has a different observation / action space)
GAE_LAMBDA = 0.99
PPO_PARAMS = dict(n_steps=2048, batch_size=256, n_epochs=5, gamma=1.0, gae_lambda=GAE_LAMBDA, clip_range=0.2,
                  ent_coef=0.005, vf_coef=0.5, max_grad_norm=0.5, target_kl=0.05, lr=3e-4,
                  net_arch_pi=[128, 128], net_arch_vf=[128, 128])
TRAIN_SIZE_PROBS = [0.40, 0.30, 0.20, 0.10]
STAGE_STEPS = 20480                     # 10 rollouts per stage
TRAIN_BUDGET_STEPS = 204800
SCRATCH_BUDGET_STEPS = 204800           # used because the warm start is incompatible
PATIENCE = 4
MIN_IMPROVEMENT = 0.02                  # in units of "multiples of plain-LSB distortion"
VAL_PAIRS_PER_SIZE = {64: 6, 128: 6, 256: 4, 512: 2}
RESUME_TRAINING = True
assert STAGE_STEPS % PPO_PARAMS["n_steps"] == 0 and PPO_PARAMS["n_steps"] % PPO_PARAMS["batch_size"] == 0

# Evaluation and noise experiments. Speckle noise is REMOVED from the active evaluation (v2 speckle files stay in the v2 folder).
EVAL_PAIRS = None
RESUME_EVAL = True
NOISE_SEED = 2024
NOISE_LEVELS = {"Gaussian": [0.25, 0.5, 1.0, 2.0],
                "Salt-and-pepper": [0.0005, 0.001, 0.005, 0.01]}
NOISE_SAVE_PAIRS = 10
SAMPLES_PER_SIZE = 3

def expected_payload_bits(secret_size=SECRET_SIZE):
    return secret_size * secret_size * 8

TOTAL_SECRET_BITS = expected_payload_bits(SECRET_SIZE)
print("Output directory:", _new_root)
print("Read-only previous experiments:", os.path.abspath(V1_ROOT), "|", os.path.abspath(OLD_ROOT))
print("Legacy layout: obs", OBS_DIM, "heads", HEAD_SIZES, "| sequential layout: obs", SEQ_OBS_DIM, "heads", SEQ_HEAD_SIZES)
print(f"PPO plan: new model, up to {SCRATCH_BUDGET_STEPS} timesteps, {STAGE_STEPS} per stage, patience {PATIENCE}")
print("Noise types:", list(NOISE_LEVELS), "| existing project files recorded:", len(EXISTING_SNAPSHOT))

In [ ]:
# STEP 3 — Dataset registration and selection. Each dataset is a list of folders that are POOLED and then re-split 80:20.
DATASET_REGISTRY = {
    "cover": {
        "DIV2K":           {"folders": [os.path.join("data", "DIV2K"), os.path.join("data", "test_covers")]},
        "Cover_Dataset_2": {"folders": [os.path.join("data", "Cover_Dataset_2", "train"), os.path.join("data", "Cover_Dataset_2", "test")]},
    },
    "secret": {
        "Secret_Existing":  {"folders": [os.path.join("data", "secret"), os.path.join("data", "test_secrets")]},
        "Secret_Dataset_2": {"folders": [os.path.join("data", "Secret_Dataset_2", "train"), os.path.join("data", "Secret_Dataset_2", "test")]},
    },
}

def register_dataset(kind, name, spec):
    DATASET_REGISTRY[kind][name] = spec

SELECTED_COVER_DATASET = "DIV2K"
SELECTED_SECRET_DATASET = "Secret_Existing"
SELECTED = {"cover": SELECTED_COVER_DATASET, "secret": SELECTED_SECRET_DATASET}

def resolve_dataset(spec):
    return {"folders": spec["folders"]} if all(os.path.isdir(f) for f in spec["folders"]) else None

print("Registered datasets:")
for kind in ("cover", "secret"):
    for name, spec in DATASET_REGISTRY[kind].items():
        print(f"  {kind:6s} | {name:18s} | {'available' if resolve_dataset(spec) else 'not available yet'}")

RESOLVED = {}
for kind in ("cover", "secret"):
    name = SELECTED[kind]
    if name not in DATASET_REGISTRY[kind]:
        raise KeyError(f"The selected {kind} dataset '{name}' is not registered.")
    RESOLVED[kind] = resolve_dataset(DATASET_REGISTRY[kind][name])
    if RESOLVED[kind] is None:
        raise FileNotFoundError(f"The selected {kind} dataset '{name}' was not found: {DATASET_REGISTRY[kind][name]}. "
                                f"No other dataset will be substituted.")
print("Selected cover dataset:", SELECTED_COVER_DATASET, "| selected secret dataset:", SELECTED_SECRET_DATASET)

In [ ]:
# STEP 4 — Pool the folders, remove exact duplicates, split 80:20 (development:test), then validation inside development
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".pgm", ".tif", ".tiff")

def list_images(folder):
    files = [f for f in os.listdir(folder) if f.lower().endswith(IMG_EXTS)]
    def key(f):
        stem = os.path.splitext(f)[0]
        return (0, int(stem), "") if stem.isdigit() else (1, 0, stem)
    return [os.path.join(folder, f) for f in sorted(files, key=key)]

def validate_image_file(path):
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            img.load()
        return True
    except Exception as e:
        raise ValueError(f"The image file is corrupted or unreadable: {path} ({e})")

stem_of = lambda p: os.path.splitext(os.path.basename(p))[0]

def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

def load_pool(kind):
    files = []
    for folder in RESOLVED[kind]["folders"]:
        files += list_images(folder)
    for f in files:
        validate_image_file(f)
    seen, kept, dropped = {}, [], []
    for f in files:
        h = file_md5(f)
        if h in seen: dropped.append((f, seen[h]))
        else: seen[h] = f; kept.append(f)
    return kept, dropped

def three_way_split(files, seed):
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(files))
    n_test = int(round(len(files) * TEST_FRACTION))
    n_val = max(1, int(round((len(files) - n_test) * VAL_FRACTION)))
    pick = lambda idx: [files[i] for i in sorted(idx.tolist())]
    return {"test": pick(order[:n_test]), "val": pick(order[n_test:n_test + n_val]), "train": pick(order[n_test + n_val:])}

cover_pool, cover_dups = load_pool("cover")
secret_pool, secret_dups = load_pool("secret")
cover_parts = three_way_split(cover_pool, GLOBAL_SEED)
secret_parts = three_way_split(secret_pool, GLOBAL_SEED + 1)

for kind, pool, dups, parts in (("cover", cover_pool, cover_dups, cover_parts), ("secret", secret_pool, secret_dups, secret_parts)):
    sets = {k: set(v) for k, v in parts.items()}
    assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"]), \
        f"Data leakage: {kind} partitions share files."
    assert sum(len(v) for v in parts.values()) == len(pool)
    dev = len(parts["train"]) + len(parts["val"])
    print(f"{kind.capitalize()} dataset ({SELECTED[kind]}): pool {len(pool)} unique images ({len(dups)} exact duplicates removed) | "
          f"train {len(parts['train'])} | validation {len(parts['val'])} | test {len(parts['test'])} | "
          f"development:test = {100*dev/len(pool):.0f}:{100*len(parts['test'])/len(pool):.0f}")

DATASET_DIRS = RESOLVED["cover"]["folders"] + RESOLVED["secret"]["folders"]
DATASET_SNAPSHOT = snapshot_files(DATASET_DIRS)
print("All dataset files are readable. Partitions are disjoint (no shared files or identical image content).")

In [ ]:
# STEP 5 — Fixed pairs. Pairs are built INSIDE each partition (train / val / test), so no cover or secret crosses partitions.
MAX_COVER_REUSE = 2      # a cover hosts at most this many DIFFERENT secrets (inside its own partition)
MAX_SECRET_REUSE = 2     # a secret is embedded in at most this many DIFFERENT covers (inside its own partition)

cover_id = {f: f"C{i+1:04d}" for i, f in enumerate(cover_pool)}
secret_id = {f: f"S{i+1:04d}" for i, f in enumerate(secret_pool)}

def pair_row(k, split, cover_file, secret_file):
    pid = f"P{k:05d}"
    r = {"pair_id": pid, "split": split, "cover_id": cover_id[cover_file], "secret_id": secret_id[secret_file],
         "cover_dataset": SELECTED_COVER_DATASET, "cover_file": cover_file,
         "secret_dataset": SELECTED_SECRET_DATASET, "secret_file": secret_file,
         "cover_path": os.path.join(PROCESSED_COVER_DIR, split, cover_id[cover_file] + ".png")}
    for s in SECRET_SIZES:
        r[f"secret_s{s}"] = os.path.join(PROCESSED_SECRET_DIR, f"s{s}", split, secret_id[secret_file] + ".png")
    return r

def make_split_pairs(covers, secrets, rng):
    n = min(MAX_SECRET_REUSE * len(secrets), MAX_COVER_REUSE * len(covers))
    cperm, sperm = rng.permutation(len(covers)), rng.permutation(len(secrets))
    pairs = [(covers[cperm[i % len(covers)]], secrets[sperm[i % len(secrets)]]) for i in range(n)]
    assert len(set(pairs)) == len(pairs), "Duplicate cover-secret pair."
    return pairs

rng_pair = np.random.RandomState(GLOBAL_SEED + 7)
records, k = [], 1
for split in ("train", "val", "test"):
    for c, s in make_split_pairs(cover_parts[split], secret_parts[split], rng_pair):
        records.append(pair_row(k, split, c, s)); k += 1
pairs_df = pd.DataFrame(records)

KEY_COLS = ["pair_id", "split", "cover_file", "secret_file"]
if os.path.exists(PAIR_FILE):
    old = pd.read_csv(PAIR_FILE)
    assert old[KEY_COLS].astype(str).equals(pairs_df[KEY_COLS].astype(str)), \
        "The saved pair assignment differs from the current datasets. Use a new ROOT folder for another dataset combination."
    print("Existing pair assignment verified and reused.")
else:
    assert_in_root(PAIR_FILE)
    pairs_df.to_csv(PAIR_FILE, index=False)
    print("Pair assignment saved:", PAIR_FILE)

all_pairs = pairs_df.to_dict("records")
train_pairs = [p for p in all_pairs if p["split"] == "train"]
val_pairs = [p for p in all_pairs if p["split"] == "val"]
test_pairs = [p for p in all_pairs if p["split"] == "test"]
pairs_by_id = {p["pair_id"]: p for p in all_pairs}

assert not pairs_df[["cover_file", "secret_file"]].duplicated().any(), "Duplicate pair."
for col in ("cover_file", "secret_file"):
    g = {sp: set(pairs_df[pairs_df.split == sp][col]) for sp in ("train", "val", "test")}
    assert not (g["train"] & g["val"]) and not (g["train"] & g["test"]) and not (g["val"] & g["test"]), f"Leakage through {col}."
assert all(p["split"] == "test" for p in test_pairs)
dev_n, tot_n = len(train_pairs) + len(val_pairs), len(all_pairs)
print(f"Pairs - train: {len(train_pairs)} | validation: {len(val_pairs)} | test: {len(test_pairs)} | "
      f"development:test = {100*dev_n/tot_n:.0f}:{100*len(test_pairs)/tot_n:.0f}")
print("Cover reuse (max pairs per cover):", int(pairs_df.cover_id.value_counts().max()),
      "| secret reuse (max pairs per secret):", int(pairs_df.secret_id.value_counts().max()))
print("Example pair:", train_pairs[0]["pair_id"], "|", os.path.basename(train_pairs[0]["cover_file"]), "+", os.path.basename(train_pairs[0]["secret_file"]))

In [ ]:
# STEP 5B — The v3 pair assignment must be identical to v2 (no silent change of splits or test pairs)
assert os.path.exists(OLD_PAIR_FILE), f"v2 pair file not found: {OLD_PAIR_FILE}"
v2_pairs = pd.read_csv(OLD_PAIR_FILE)
cols = ["pair_id", "split", "cover_file", "secret_file"]
assert v2_pairs[cols].astype(str).equals(pairs_df[cols].astype(str)), "The v3 pair assignment differs from v2."
print(f"Pair assignment identical to v2: train {len(train_pairs)} | validation {len(val_pairs)} | test {len(test_pairs)}")

In [ ]:
# STEP 6 — Preprocessing. Processed files are written only to the new directory. Identical files from the previous
# experiment are COPIED (the old files are only read), which avoids repeating about an hour of preprocessing.
norm = os.path.normpath
OLD_COVER_MAP, OLD_SECRET_MAP = {}, {}
if os.path.exists(OLD_PAIR_FILE):
    _old = pd.read_csv(OLD_PAIR_FILE)
    for r in _old.to_dict("records"):
        OLD_COVER_MAP[norm(r["cover_file"])] = r
        OLD_SECRET_MAP[norm(r["secret_file"])] = r
print("Previous processed images available for reuse: covers", len(OLD_COVER_MAP), "| secrets", len(OLD_SECRET_MAP))

def copy_or_make(dst, old_src, make_fn):
    if os.path.exists(dst): return "exists"
    assert_in_root(dst); os.makedirs(os.path.dirname(dst), exist_ok=True)
    if old_src and os.path.exists(old_src):
        shutil.copyfile(old_src, dst); return "reused"
    make_fn(dst); return "new"

def make_cover(src, dst):
    with Image.open(src) as im:
        im.convert("RGB").resize((COVER_SIZE, COVER_SIZE), Image.BICUBIC).save(dst)

def make_secret(src, dst, size):
    with Image.open(src) as im:
        im.convert("L").resize((size, size), Image.BICUBIC).save(dst)

t0 = time.perf_counter(); stats = Counter(); done = set(); unique_files = []
for p in all_pairs:
    if p["cover_path"] not in done:
        done.add(p["cover_path"]); unique_files.append((p["cover_path"], "cover", None))
        old = OLD_COVER_MAP.get(norm(p["cover_file"]))
        stats["cover " + copy_or_make(p["cover_path"], norm(old["cover_path"]) if old else None,
                                      lambda d, f=p["cover_file"]: make_cover(f, d))] += 1
    old_s = OLD_SECRET_MAP.get(norm(p["secret_file"]))
    for s in SECRET_SIZES:
        dst = p[f"secret_s{s}"]
        if dst in done: continue
        done.add(dst); unique_files.append((dst, "secret", s))
        stats["secret " + copy_or_make(dst, norm(old_s[f"secret_s{s}"]) if old_s else None,
                                       lambda d, f=p["secret_file"], z=s: make_secret(f, d, z))] += 1
print(f"Preprocessing completed in {time.perf_counter()-t0:.0f} s:", dict(stats))
for path, kind, s in unique_files:
    with Image.open(path) as im:
        if kind == "cover": assert im.mode == "RGB" and im.size == (COVER_SIZE, COVER_SIZE), f"Bad cover: {path}"
        else: assert im.mode == "L" and im.size == (s, s), f"Bad secret: {path}"
print(f"Verified {len(unique_files)} processed files: covers {COVER_SIZE}x{COVER_SIZE} RGB, secrets grayscale at {SECRET_SIZES}.")

In [ ]:
# STEP 6B — Copy v2 block-feature caches (read only source); STEP 14 still validates each file with its checksum
copied = missing = 0
for cid in {p["cover_id"] for p in all_pairs}:
    name = f"{cid}_c{COVER_SIZE}_b{BLOCK_SIZE}.npz"
    src, dst = os.path.join(OLD_CACHE_DIR, name), os.path.join(CACHE_DIR, name)
    if os.path.exists(dst):
        continue
    if os.path.exists(src):
        assert_in_root(dst); shutil.copyfile(src, dst); copied += 1
    else:
        missing += 1
print(f"Feature caches copied from v2: {copied} | missing (will be recomputed): {missing}")

In [ ]:
# STEP 7 — Load and validate images
def load_cover_image(path):
    with Image.open(path) as img:
        arr = np.array(img.convert("RGB"), dtype=np.uint8)
    assert arr.shape == (COVER_SIZE, COVER_SIZE, 3)
    return arr

def load_secret_image(path, size):
    with Image.open(path) as img:
        arr = np.array(img.convert("L"), dtype=np.uint8)
    assert arr.shape == (size, size)
    return arr

def validate_image(arr, shape, name):
    assert arr.shape == shape, f"{name} has an unexpected shape."
    assert arr.dtype == np.uint8, f"{name} has an unexpected data type."
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, pixel range={arr.min()}-{arr.max()}")

DEMO_PAIR = train_pairs[0]
cover_image = load_cover_image(DEMO_PAIR["cover_path"])
secret_image = load_secret_image(DEMO_PAIR[f"secret_s{SECRET_SIZE}"], SECRET_SIZE)
validate_image(cover_image, (COVER_SIZE, COVER_SIZE, 3), "Cover image")
validate_image(secret_image, (SECRET_SIZE, SECRET_SIZE), "Secret image")

In [ ]:
# STEP 8 — Secret image to binary
secret_bits = np.unpackbits(secret_image.flatten())
assert secret_bits.shape[0] == TOTAL_SECRET_BITS == secret_image.size * 8
assert np.array_equal(np.packbits(secret_bits).reshape(secret_image.shape), secret_image)
print("Secret image converted to binary:", secret_bits.shape[0], "bits")

In [ ]:
# STEP 9 — DNA mapping rules (unchanged)
dna_mapping_rules = [
    {0: 'A', 1: 'C', 2: 'G', 3: 'T'}, {0: 'A', 2: 'C', 1: 'G', 3: 'T'},
    {1: 'A', 0: 'C', 3: 'G', 2: 'T'}, {1: 'A', 3: 'C', 0: 'G', 2: 'T'},
    {2: 'A', 0: 'C', 3: 'G', 1: 'T'}, {2: 'A', 3: 'C', 0: 'G', 1: 'T'},
    {3: 'A', 1: 'C', 2: 'G', 0: 'T'}, {3: 'A', 2: 'C', 1: 'G', 0: 'T'},
]
assert len(dna_mapping_rules) == NUM_DNA_RULES
dna_reverse_rules = [{v: k for k, v in rule.items()} for rule in dna_mapping_rules]
for i, rule in enumerate(dna_mapping_rules):
    assert sorted(rule.keys()) == [0, 1, 2, 3] and sorted(rule.values()) == ['A', 'C', 'G', 'T']
    assert all(dna_reverse_rules[i][rule[v]] == v for v in range(4))
assert len({tuple(sorted(r.items())) for r in dna_mapping_rules}) == NUM_DNA_RULES
BASE_DNA_RULE = 0
for i, rule in enumerate(dna_mapping_rules):
    print(f"  Rule {i}: 00->{rule[0]}, 01->{rule[1]}, 10->{rule[2]}, 11->{rule[3]}")
print("All 8 DNA mapping rules are one-to-one, distinct and reversible.")

In [ ]:
# STEP 10 — Binary bits <-> DNA symbol values and letters
def bits_to_symbol_values(bits):
    bits = np.asarray(bits, dtype=np.uint8)
    assert len(bits) % 2 == 0
    pairs = bits.reshape(-1, 2)
    return ((pairs[:, 0] << 1) | pairs[:, 1]).astype(np.uint8)

def symbol_values_to_bits(values):
    values = np.asarray(values, dtype=np.uint8)
    bits = np.zeros(len(values) * 2, dtype=np.uint8)
    bits[0::2] = (values >> 1) & 1
    bits[1::2] = values & 1
    return bits

def symbols_to_dna_string(values, rule_idx):
    return "".join(dna_mapping_rules[rule_idx][int(v)] for v in values)

def dna_string_to_symbols(dna, rule_idx):
    return np.array([dna_reverse_rules[rule_idx][c] for c in dna], dtype=np.uint8)

secret_symbol_values = bits_to_symbol_values(secret_bits)
assert np.array_equal(symbol_values_to_bits(secret_symbol_values), secret_bits)
dna_text = symbols_to_dna_string(secret_symbol_values[:16], BASE_DNA_RULE)
assert np.array_equal(dna_string_to_symbols(dna_text, BASE_DNA_RULE), secret_symbol_values[:16])
print("DNA symbols for the example secret:", len(secret_symbol_values), "| first 16 bases:", dna_text)

In [ ]:
# STEP 11 — Divide the cover image into 8x8 blocks
def image_to_blocks(image, block_size=BLOCK_SIZE):
    H, W, C = image.shape
    return (image.reshape(H // block_size, block_size, W // block_size, block_size, C)
                 .swapaxes(1, 2).reshape(-1, block_size, block_size, C))

def blocks_to_image(blocks_, image_shape, block_size=BLOCK_SIZE):
    nb_r, nb_c = image_shape[0] // block_size, image_shape[1] // block_size
    return (blocks_.reshape(nb_r, nb_c, block_size, block_size, image_shape[2])
                   .swapaxes(1, 2).reshape(image_shape))

blocks = image_to_blocks(cover_image, BLOCK_SIZE)
assert blocks.shape == (NUM_BLOCKS, BLOCK_SIZE, BLOCK_SIZE, 3)
assert np.array_equal(blocks_to_image(blocks, cover_image.shape, BLOCK_SIZE), cover_image)
print("Cover divided into blocks. Block array shape:", blocks.shape, "(reconstruction verified)")

In [ ]:
# STEP 12 — Pixel locations for embedding (row-major)
pixel_locations = [(r, c) for r in range(BLOCK_SIZE) for c in range(BLOCK_SIZE)]
assert len(pixel_locations) == NUM_PIXEL_LOCATIONS
print("Pixel locations per block:", len(pixel_locations))

In [ ]:
# STEP 13 — Block features: variance, entropy, Canny edge density, GLCM contrast (computed on the LSB-masked block)
def calculate_variance(gray):
    return float(np.var(gray.astype(np.float64)))

def calculate_entropy(gray):
    hist, _ = np.histogram(gray, bins=256, range=(0, 255))
    probs = hist / (hist.sum() + 1e-12)
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))

def calculate_edge_density(gray):
    edges = cv2.Canny(gray, 50, 150)
    return float(np.mean(edges > 0))

def calculate_glcm_contrast(gray, levels=8):
    quantized = np.clip((gray // (256 // levels)).astype(np.uint8), 0, levels - 1)
    glcm = graycomatrix(quantized, distances=[1], angles=[0], levels=levels, symmetric=True, normed=True)
    return float(graycoprops(glcm, 'contrast')[0, 0])

def get_block_features(rgb_block):
    masked = (rgb_block & np.uint8(FEATURE_LSB_MASK)).astype(np.uint8)
    gray = cv2.cvtColor(masked, cv2.COLOR_RGB2GRAY)
    return np.array([calculate_variance(gray), calculate_entropy(gray),
                     calculate_edge_density(gray), calculate_glcm_contrast(gray)], dtype=np.float64)

print("Example feature vector:", get_block_features(blocks[0]))

In [ ]:
# STEP 14 — Min-max normalisation, per-cover feature cache (previous cache files are copied, validated by checksum) and sample preparation
def normalize_features(raw):
    fmin, fmax = raw.min(axis=0), raw.max(axis=0)
    frange = np.where((fmax - fmin) == 0, 1.0, fmax - fmin)
    return np.clip((raw - fmin) / frange, 0.0, 1.0).astype(np.float32), fmin, fmax, frange

_feature_mem_cache = {}

def get_cover_features(pair, cover):
    cid = pair["cover_id"]
    if cid in _feature_mem_cache:
        return _feature_mem_cache[cid]
    cache_file = os.path.join(CACHE_DIR, f"{cid}_c{COVER_SIZE}_b{BLOCK_SIZE}.npz")
    if not os.path.exists(cache_file):
        old = OLD_COVER_MAP.get(os.path.normpath(pair["cover_file"]))
        if old is not None:
            oc = os.path.join(OLD_CACHE_DIR, f"{old['pair_id']}_c{COVER_SIZE}_b{BLOCK_SIZE}.npz")
            if os.path.exists(oc):
                shutil.copyfile(oc, cache_file)
    checksum = int(cover.sum(dtype=np.int64))
    raw, feat_time = None, float("nan")
    if os.path.exists(cache_file):
        try:
            with np.load(cache_file) as d:
                if int(d["checksum"]) == checksum and d["raw"].shape == (NUM_BLOCKS, NUM_FEATURES):
                    raw, feat_time = d["raw"].copy(), float(d["feat_time"])
        except Exception:
            raw = None
    if raw is None:
        t0 = time.perf_counter()
        raw = np.array([get_block_features(b) for b in image_to_blocks(cover, BLOCK_SIZE)], dtype=np.float64)
        feat_time = time.perf_counter() - t0
        np.savez(cache_file, raw=raw, feat_time=feat_time, checksum=checksum)
    assert np.isfinite(raw).all()
    state, _, _, _ = normalize_features(raw)
    assert state.dtype == np.float32 and 0.0 <= state.min() <= state.max() <= 1.0
    _feature_mem_cache[cid] = (state, feat_time)
    return _feature_mem_cache[cid]

def load_pair_arrays(pair, size):
    cover = load_cover_image(pair["cover_path"])
    secret_img = load_secret_image(pair[f"secret_s{size}"], size)
    return cover, secret_img, np.unpackbits(secret_img.reshape(-1))

def prepare_sample(pair, size=SECRET_SIZE):
    cover, secret_img, sbits = load_pair_arrays(pair, size)
    assert len(sbits) == expected_payload_bits(size) and len(sbits) % 2 == 0
    state, feat_time = get_cover_features(pair, cover)
    return {"pair": pair, "secret_size": int(size), "cover_image": cover, "secret_image": secret_img,
            "blocks": image_to_blocks(cover, BLOCK_SIZE), "secret_bits": sbits, "payload_bits": sbits.copy(),
            "state_matrix": state, "feat_time": feat_time}

print("Extracting / loading block features for all covers...")
t0 = time.perf_counter()
_unique_cover_pairs = {}
for p in all_pairs:
    _unique_cover_pairs.setdefault(p["cover_id"], p)
for i, p in enumerate(_unique_cover_pairs.values()):
    get_cover_features(p, load_cover_image(p["cover_path"]))
    if (i + 1) % 100 == 0 or i + 1 == len(_unique_cover_pairs):
        print(f"  {i+1}/{len(_unique_cover_pairs)} covers | elapsed {time.perf_counter()-t0:.0f} s")
_chk = prepare_sample(train_pairs[0], 64)
assert _chk["state_matrix"].shape == (NUM_BLOCKS, NUM_FEATURES) and _chk["payload_bits"].shape[0] == 64 * 64 * 8
print("Feature cache ready:", CACHE_DIR)

In [ ]:
# STEP 15 — Payload (raw secret bits; the DNA transformation is applied per block during embedding) and capacity
payload_bits = secret_bits.copy()
assert len(payload_bits) == TOTAL_SECRET_BITS and len(payload_bits) % 2 == 0
cap_rows = []
for s in SECRET_SIZES:
    bits = expected_payload_bits(s)
    cap_rows.append({"Secret size": f"{s} x {s}", "Payload (bits)": bits, "Payload rate (bpp)": bits / (COVER_SIZE * COVER_SIZE),
                     "Cover capacity (bits)": THEORETICAL_MAX_CAPACITY_BITS,
                     "Payload / capacity (%)": round(100.0 * bits / THEORETICAL_MAX_CAPACITY_BITS, 2),
                     "Minimum blocks (3 LSB)": math.ceil(bits / MAX_BLOCK_BITS)})
print(pd.DataFrame(cap_rows).to_string(index=False))

In [ ]:
# STEP 16 — DNA rule transformation (bits re-encoded from the base rule to the chosen rule, and back)
dna_transform_lut = np.zeros((NUM_DNA_RULES, NUM_DNA_RULES, 4), dtype=np.uint8)
for s_ in range(NUM_DNA_RULES):
    for d_ in range(NUM_DNA_RULES):
        for v in range(4):
            dna_transform_lut[s_, d_, v] = dna_reverse_rules[d_][dna_mapping_rules[s_][v]]

def apply_dna_rule_transform(bits, src_rule_idx, dst_rule_idx):
    bits = np.asarray(bits, dtype=np.uint8).copy()
    n_pairs = len(bits) // 2
    if n_pairs > 0:
        pairs = bits[:2 * n_pairs].reshape(-1, 2)
        values = (pairs[:, 0] << 1) | pairs[:, 1]
        new_values = dna_transform_lut[src_rule_idx, dst_rule_idx][values]
        bits[0:2 * n_pairs:2] = (new_values >> 1) & 1
        bits[1:2 * n_pairs:2] = new_values & 1
    return bits

rng_t = np.random.RandomState(GLOBAL_SEED)
test_bits = rng_t.randint(0, 2, size=1000).astype(np.uint8)
for r in range(NUM_DNA_RULES):
    fwd = apply_dna_rule_transform(test_bits, BASE_DNA_RULE, r)
    assert np.array_equal(apply_dna_rule_transform(fwd, r, BASE_DNA_RULE), test_bits)
for s_ in range(NUM_DNA_RULES):
    for d_ in range(NUM_DNA_RULES):
        assert sorted(dna_transform_lut[s_, d_].tolist()) == [0, 1, 2, 3]
print("DNA transformation and reverse transformation verified for all 8 rules.")

In [ ]:
# STEP 17 — Adaptive LSB embedding inside one block
def embed_action_in_block(original_block, remaining_payload_bits, dna_rule_id, lsb_amount,
                          location_id, pixel_locations_, base_dna_rule=BASE_DNA_RULE, span=EMBED_SPAN):
    stego_block = original_block.copy()
    n_bits = min(span * 3 * lsb_amount, len(remaining_payload_bits))
    if n_bits <= 0:
        return stego_block, 0
    bits = apply_dna_rule_transform(remaining_payload_bits[:n_bits], base_dna_rule, dna_rule_id)
    n_slots = -(-n_bits // lsb_amount)
    padded = np.zeros(n_slots * lsb_amount, dtype=np.int64)
    padded[:n_bits] = bits
    weights = 1 << np.arange(lsb_amount - 1, -1, -1)
    slot_values = padded.reshape(n_slots, lsb_amount) @ weights
    pl = np.asarray(pixel_locations_)
    s = np.arange(n_slots)
    pix = (location_id + s // 3) % len(pl)
    rows, cols, chans = pl[pix, 0], pl[pix, 1], s % 3
    cur = stego_block[rows, cols, chans].astype(np.int64)
    keep_mask = 0xFF ^ ((1 << lsb_amount) - 1)
    stego_block[rows, cols, chans] = ((cur & keep_mask) | slot_values).astype(np.uint8)
    return stego_block, int(n_bits)

print("Embedding function defined.")

In [ ]:
# STEP 18 — Extraction inside one block and round-trip validation
def extract_bits_from_block(stego_block, lsb_amount, location_id, pixel_locations_, n_bits_needed, span=EMBED_SPAN):
    if n_bits_needed <= 0:
        return np.zeros(0, dtype=np.uint8)
    n_slots = -(-n_bits_needed // lsb_amount)
    assert n_slots <= span * 3, "The requested number of bits exceeds the block capacity."
    pl = np.asarray(pixel_locations_)
    s = np.arange(n_slots)
    pix = (location_id + s // 3) % len(pl)
    vals = stego_block[pl[pix, 0], pl[pix, 1], s % 3].astype(np.int64) & ((1 << lsb_amount) - 1)
    shifts = np.arange(lsb_amount - 1, -1, -1)
    bits = ((vals[:, None] >> shifts) & 1).reshape(-1)
    return bits[:n_bits_needed].astype(np.uint8)

def recover_bits_from_block(stego_block, dna_rule_id, lsb_amount, location_id, pixel_locations_,
                            n_bits_needed, base_dna_rule=BASE_DNA_RULE, span=EMBED_SPAN):
    raw = extract_bits_from_block(stego_block, lsb_amount, location_id, pixel_locations_, n_bits_needed, span)
    return apply_dna_rule_transform(raw, dna_rule_id, base_dna_rule)

checked = 0
for bi in (0, NUM_BLOCKS // 2, NUM_BLOCKS - 1):
    blk = blocks[bi]
    for rule in range(NUM_DNA_RULES):
        for lsb in (1, 2, 3):
            cap = EMBED_SPAN * 3 * lsb
            for loc in (0, 10, NUM_PIXEL_LOCATIONS - 1):
                for n in (2, 50, cap - 2, cap, cap + 400):
                    seg = rng_t.randint(0, 2, size=n).astype(np.uint8)
                    stego, used = embed_action_in_block(blk, seg, rule, lsb, loc, pixel_locations)
                    assert used == min(n, cap)
                    assert np.array_equal(recover_bits_from_block(stego, rule, lsb, loc, pixel_locations, used), seg[:used])
                    assert np.all((np.bitwise_xor(blk, stego) >> lsb) == 0)
                    checked += 1
assert np.array_equal(get_block_features(blocks[0]), get_block_features(
    embed_action_in_block(blocks[0], payload_bits[:600], 3, 3, 10, pixel_locations)[0]))
print(f"Embedding and extraction verified: {checked} round trips passed; block features are unchanged by embedding.")

In [ ]:
# STEP 19 — Metrics
def calculate_mse(img1, img2):
    return float(np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2))

def calculate_psnr(img1, img2, max_pixel=255.0):
    mse = calculate_mse(img1, img2)
    return float('inf') if mse == 0 else float(20 * np.log10(max_pixel / np.sqrt(mse)))

def calculate_ssim(img1, img2):
    if img1.ndim == 3:
        return float(ssim_metric(img1, img2, channel_axis=2, data_range=255))
    return float(ssim_metric(img1, img2, data_range=255))

def fast_block_ssim(a, b):
    x = a.reshape(-1, a.shape[-1]).astype(np.float64)
    y = b.reshape(-1, b.shape[-1]).astype(np.float64)
    mx, my = x.mean(0), y.mean(0)
    vx, vy = x.var(0), y.var(0)
    cxy = ((x - mx) * (y - my)).mean(0)
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    s = ((2 * mx * my + c1) * (2 * cxy + c2)) / ((mx ** 2 + my ** 2 + c1) * (vx + vy + c2))
    return float(s.mean())

def bit_metrics(orig_bits, rec_bits):
    same = len(orig_bits) == len(rec_bits)
    acc = float(np.mean(np.asarray(orig_bits) == np.asarray(rec_bits))) if same else 0.0
    return {"ber": 1.0 - acc, "bit_accuracy": acc, "exact_recovery": bool(same and acc == 1.0)}

assert calculate_mse(cover_image, cover_image) == 0 and calculate_psnr(cover_image, cover_image) == float("inf")
assert bit_metrics(secret_bits, secret_bits)["exact_recovery"]
print("Metric functions defined: MSE, PSNR, SSIM (cover vs stego), BER, bit accuracy and exact recovery (secret).")

In [ ]:
# STEP 19B — Image-level quality of a recovered secret (kept separate from BER / exact recovery)
def secret_quality(orig_img, rec_img):
    return {"secret_mse": calculate_mse(orig_img, rec_img), "secret_psnr": calculate_psnr(orig_img, rec_img),
            "secret_ssim": calculate_ssim(orig_img, rec_img), "secret_pixel_accuracy": float(np.mean(orig_img == rec_img))}

_q = secret_quality(secret_image, secret_image)
assert _q["secret_mse"] == 0 and _q["secret_pixel_accuracy"] == 1.0 and abs(_q["secret_ssim"] - 1.0) < 1e-9
print("Secret quality helper defined: MSE, PSNR, SSIM and pixel accuracy of the recovered secret.")

In [ ]:
# STEP 20 — Reward function (unchanged)
def calculate_block_quality_distortion(original_block, stego_block, flip_rate):
    mse = calculate_mse(original_block, stego_block)
    psnr = PSNR_CEIL_DB if mse == 0 else min(20 * np.log10(255.0 / np.sqrt(mse)), PSNR_CEIL_DB)
    psnr_term = float(np.clip((psnr - PSNR_FLOOR_DB) / (PSNR_CEIL_DB - PSNR_FLOOR_DB), 0.0, 1.0))
    ssim_term = 1.0 if mse == 0 else float(np.clip(fast_block_ssim(original_block, stego_block), 0.0, 1.0))
    return 0.5 * psnr_term + 0.5 * ssim_term, 1.0 - float(np.clip(flip_rate, 0.0, 1.0))

def calculate_reward(original_block, stego_block, bits_embedded, total_bits, lsb_amount=1,
                     texture_score=1.0, flip_rate=0.0, is_skip=False, payload_incomplete=True):
    if is_skip:
        return float(-SKIP_PENALTY) if payload_incomplete else 0.0
    if bits_embedded <= 0 or total_bits <= 0:
        return float(-SKIP_PENALTY)
    quality, distortion = calculate_block_quality_distortion(original_block, stego_block, flip_rate)
    frac = bits_embedded / total_bits
    lsb_penalty = LSB_PENALTY_WEIGHT * (1.0 - texture_score) * (lsb_amount - 1)
    reward = frac * (PROGRESS_WEIGHT + QUALITY_WEIGHT * quality + DISTORTION_WEIGHT * distortion - lsb_penalty)
    if not np.isfinite(reward):
        reward = -STEP_REWARD_CLIP
    return float(np.clip(reward, -STEP_REWARD_CLIP, STEP_REWARD_CLIP))

def embedding_reward(original_block, stego_block, bits_used, lsb_amount, location_id, pixel_locations_,
                     total_bits, texture_score, payload_incomplete):
    if bits_used > 0:
        ob = extract_bits_from_block(original_block, lsb_amount, location_id, pixel_locations_, bits_used)
        sb = extract_bits_from_block(stego_block, lsb_amount, location_id, pixel_locations_, bits_used)
        flip_rate = float(np.mean(ob != sb))
    else:
        flip_rate = 0.0
    return calculate_reward(original_block, stego_block, bits_used, total_bits, lsb_amount, texture_score,
                            flip_rate, is_skip=False, payload_incomplete=payload_incomplete)

_sb1, _bu1 = embed_action_in_block(blocks[0], payload_bits[:192], 3, 1, 10, pixel_locations)
_sb3, _bu3 = embed_action_in_block(blocks[0], payload_bits[:576], 3, 3, 10, pixel_locations)
_r1 = embedding_reward(blocks[0], _sb1, _bu1, 1, 10, pixel_locations, TOTAL_SECRET_BITS, 0.5, True)
_r3 = embedding_reward(blocks[0], _sb3, _bu3, 3, 10, pixel_locations, TOTAL_SECRET_BITS, 0.5, True)
_rs = calculate_reward(blocks[0], blocks[0], 0, TOTAL_SECRET_BITS, is_skip=True, payload_incomplete=True)
assert _r1 > _rs and np.isfinite([_r1, _r3, _rs]).all() and _r1 / _bu1 > _r3 / _bu3
print(f"Reward check passed: 1-LSB {_r1:.6f} | 3-LSB {_r3:.6f} | skip {_rs:.6f}")

In [ ]:
# STEP 21 — Gymnasium environment (DNASteganographyEnv). One pair is always used with its own assigned secret.
def build_observation(state_matrix, used_mask, payload_ptr, total_bits):
    remaining = max(0, total_bits - payload_ptr)
    n_unused = int((~used_mask).sum())
    extra = np.array([payload_ptr / total_bits,
                      min(1.0, remaining / max(1, n_unused * MAX_BLOCK_BITS)),
                      total_bits / THEORETICAL_MAX_CAPACITY_BITS], dtype=np.float32)
    obs = np.concatenate([state_matrix.reshape(-1), used_mask.astype(np.float32), extra]).astype(np.float32)
    return np.clip(obs, 0.0, 1.0)

class DNASteganographyEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, pairs, secret_sizes, pixel_locations_, size_probs=None, use_reward=True,
                 use_feasibility_mask=USE_FEASIBILITY_MASK, base_dna_rule=BASE_DNA_RULE,
                 max_episode_steps=MAX_EPISODE_STEPS):
        super().__init__()
        self.pairs, self.secret_sizes = list(pairs), list(secret_sizes)
        p = np.asarray(size_probs if size_probs is not None else np.ones(len(self.secret_sizes)), dtype=np.float64)
        self.size_probs = p / p.sum()
        self.pixel_locations, self.base_dna_rule = pixel_locations_, base_dna_rule
        self.use_reward, self.use_feasibility_mask = use_reward, use_feasibility_mask
        self.max_episode_steps = max_episode_steps
        self.observation_space = spaces.Box(0.0, 1.0, (OBS_DIM,), dtype=np.float32)
        self.action_space = spaces.MultiDiscrete(HEAD_SIZES)
        self.rng = np.random.RandomState(GLOBAL_SEED)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.RandomState(seed)
        if options and "sample" in options:
            sample = options["sample"]
        else:
            pair = self.pairs[int(self.rng.randint(len(self.pairs)))]
            size = int(self.rng.choice(self.secret_sizes, p=self.size_probs))
            sample = prepare_sample(pair, size)
        self.sample = sample
        self.cover_blocks, self.stego_blocks = sample["blocks"], sample["blocks"].copy()
        self.state_matrix, self.payload_bits = sample["state_matrix"], sample["payload_bits"]
        self.total_bits, self.secret_size = len(self.payload_bits), sample["secret_size"]
        self.used_mask = np.zeros(NUM_BLOCKS, dtype=bool)
        self.block_bit_log = {}
        self.payload_ptr = 0; self.step_count = 0; self.invalid_picks = 0; self.episode_return = 0.0
        return build_observation(self.state_matrix, self.used_mask, self.payload_ptr, self.total_bits), {}

    def action_masks(self):
        n_unused = int((~self.used_mask).sum())
        remaining = int(self.total_bits - self.payload_ptr)
        block_mask = ~self.used_mask
        dna_mask = np.ones(NUM_DNA_RULES, dtype=bool)
        lsb_mask = np.ones(NUM_LSB_OPTIONS, dtype=bool)
        loc_mask = np.ones(NUM_PIXEL_LOCATIONS, dtype=bool)
        embed_mask = np.ones(NUM_EMBED_FLAGS, dtype=bool)
        if self.use_feasibility_mask and remaining > 0 and n_unused > 0:
            cap_other = (n_unused - 1) * MAX_BLOCK_BITS
            for k in range(NUM_LSB_OPTIONS):
                lsb_mask[k] = (cap_other + EMBED_SPAN * 3 * (k + 1)) >= remaining
            embed_mask[0] = cap_other >= remaining
            if not lsb_mask.any():
                lsb_mask[-1] = True
        return np.concatenate([block_mask, dna_mask, lsb_mask, loc_mask, embed_mask])

    def step(self, action):
        block_id = int(action[0])
        self.step_count += 1
        if self.used_mask[block_id]:                      # unreachable under correct masking
            self.invalid_picks += 1
            completed = self.payload_ptr >= self.total_bits
            terminated = completed or bool(np.all(self.used_mask))
            truncated = (not terminated) and (self.step_count >= self.max_episode_steps)
            obs = build_observation(self.state_matrix, self.used_mask, self.payload_ptr, self.total_bits)
            reward = -INVALID_PICK_PENALTY if self.use_reward else 0.0
            self.episode_return += reward
            return obs, float(reward), terminated, truncated, {"invalid_pick": True, "block_id": block_id, "bits_embedded": 0,
                    "payload_ptr": int(self.payload_ptr), "secret_size": self.secret_size, "completed": bool(completed)}

        dna_rule_id, lsb_amount = int(action[1]), int(action[2]) + 1
        location_id, embed_flag = int(action[3]), int(action[4])
        original_block = self.cover_blocks[block_id]
        remaining = self.payload_bits[self.payload_ptr:]
        texture_score = float(np.mean(self.state_matrix[block_id]))
        payload_incomplete = self.payload_ptr < self.total_bits
        reward = 0.0

        if embed_flag == 1 and len(remaining) > 0:
            stego_block, bits_used = embed_action_in_block(original_block, remaining, dna_rule_id, lsb_amount,
                                                           location_id, self.pixel_locations, self.base_dna_rule)
            self.stego_blocks[block_id] = stego_block
            self.block_bit_log[block_id] = {"embedded": True, "dna_rule": dna_rule_id, "lsb": lsb_amount,
                                            "location": location_id, "bits": int(bits_used),
                                            "payload_start": int(self.payload_ptr)}
            if self.use_reward:
                reward = embedding_reward(original_block, stego_block, bits_used, lsb_amount, location_id,
                                          self.pixel_locations, self.total_bits, texture_score, payload_incomplete)
        else:
            bits_used = 0
            self.block_bit_log[block_id] = {"embedded": False, "dna_rule": None, "lsb": None, "location": None,
                                            "bits": 0, "payload_start": int(self.payload_ptr)}
            if self.use_reward:
                reward = calculate_reward(original_block, original_block, 0, self.total_bits,
                                          is_skip=True, payload_incomplete=payload_incomplete)

        self.used_mask[block_id] = True
        self.payload_ptr += bits_used
        completed = self.payload_ptr >= self.total_bits
        terminated = completed or bool(np.all(self.used_mask))
        truncated = (not terminated) and (self.step_count >= self.max_episode_steps)

        if self.use_reward:
            if completed:
                reward += COMPLETION_BONUS
            elif terminated or truncated:
                reward -= INCOMPLETE_PENALTY_SCALE * (self.total_bits - self.payload_ptr) / self.total_bits
            reward = float(np.clip(reward, -STEP_REWARD_CLIP - INCOMPLETE_PENALTY_SCALE, STEP_REWARD_CLIP + COMPLETION_BONUS))
            assert np.isfinite(reward)
        self.episode_return += reward

        obs = build_observation(self.state_matrix, self.used_mask, self.payload_ptr, self.total_bits)
        info = {"bits_embedded": int(bits_used), "payload_ptr": int(self.payload_ptr), "block_id": block_id,
                "secret_size": self.secret_size, "completed": bool(completed)}
        return obs, float(reward), terminated, truncated, info

def _mask_fn(env_):
    return env_.action_masks()

print("DNASteganographyEnv defined. Observation dimension:", OBS_DIM)

In [ ]:
# STEP 22 — Policies, episode runner and embedding-record checker
def split_action_masks(masks):
    masks = np.asarray(masks)
    return [masks[HEAD_OFFSETS[i]:HEAD_OFFSETS[i + 1]] for i in range(len(HEAD_SIZES))]

class PPOChooser:
    """Wraps the trained PPO policy; every embedding decision comes from model.predict()."""
    def __init__(self, model_):
        self.model, self.calls = model_, 0
    def __call__(self, obs, masks):
        action, _ = self.model.predict(obs, action_masks=masks, deterministic=True)
        self.calls += 1
        return action

def make_ppo_chooser(model_):
    return PPOChooser(model_)

def make_random_valid_chooser(seed):
    rng = np.random.RandomState(seed)
    def chooser(obs, masks):
        return np.array([rng.choice(np.flatnonzero(m)) for m in split_action_masks(masks)], dtype=np.int64)
    return chooser

def run_embedding_episode(sample, chooser, use_reward=False):
    env = DNASteganographyEnv([sample["pair"]], [sample["secret_size"]], pixel_locations, use_reward=use_reward)
    obs, _ = env.reset(options={"sample": sample})
    done, steps = False, 0
    while not done:
        obs, _, terminated, truncated, _ = env.step(chooser(obs, env.action_masks()))
        done = terminated or truncated
        steps += 1
    return env, steps

def records_from_log(block_bit_log):
    recs = [{"block_index": int(b), "dna_rule": int(l["dna_rule"]), "lsb_depth": int(l["lsb"]),
             "pixel_location": int(l["location"]), "bits_embedded": int(l["bits"]), "payload_start": int(l["payload_start"])}
            for b, l in block_bit_log.items() if l["embedded"] and l["bits"] > 0]
    recs.sort(key=lambda r: r["payload_start"])
    return recs

def check_records(records, total_bits):
    expected, seen = 0, set()
    for r in records:
        if r["block_index"] in seen: return False, "block reused"
        seen.add(r["block_index"])
        if r["payload_start"] != expected: return False, f"gap or overlap at payload position {expected}"
        if not (1 <= r["lsb_depth"] <= MAX_LSB): return False, "bad LSB depth"
        if not (0 < r["bits_embedded"] <= EMBED_SPAN * 3 * r["lsb_depth"]): return False, "bad bit count"
        expected += r["bits_embedded"]
    if expected != total_bits: return False, f"records sum to {expected}, payload is {total_bits}"
    return True, "ok"

print("Policies, episode runner and record checker defined.")

In [ ]:
# STEP 23 — Environment validation for all secret sizes (random valid policy)
check_env(ActionMasker(DNASteganographyEnv(train_pairs[:3], [64], pixel_locations), _mask_fn), warn=True)
print("Gymnasium environment check passed.")
rows = []
for size in SECRET_SIZES:
    smp = prepare_sample(train_pairs[0], size)
    total = len(smp["payload_bits"])
    env, steps = run_embedding_episode(smp, make_random_valid_chooser(GLOBAL_SEED))
    recs = records_from_log(env.block_bit_log)
    ok, msg = check_records(recs, total); assert ok, msg
    assert env.payload_ptr == total == sum(r["bits_embedded"] for r in recs) and env.invalid_picks == 0
    assert len(env.block_bit_log) == int(env.used_mask.sum()) == steps
    bits = np.concatenate([recover_bits_from_block(env.stego_blocks[r["block_index"]], r["dna_rule"], r["lsb_depth"],
                           r["pixel_location"], pixel_locations, r["bits_embedded"]) for r in recs])
    assert np.array_equal(bits, smp["secret_bits"])
    rows.append({"secret size": size, "payload bits": total, "steps": steps, "blocks embedded": len(recs), "completed": True})
print(pd.DataFrame(rows).to_string(index=False))
print("Environment checks passed: no block reuse, complete payload, exact bit recovery for all sizes.")

In [ ]:
# STEP 23B — Audit of the v2 reward and state on validation pairs (legacy environment)
def legacy_fixed_depth_chooser(depth):
    def chooser(obs, masks):
        parts = split_action_masks(masks)
        s = int(np.flatnonzero(parts[0])[0])                         # first unused block (raster order)
        ok, want = np.flatnonzero(parts[2]), depth - 1
        d = want if parts[2][want] else (int(ok[ok >= want][0]) if (ok >= want).any() else int(ok[-1]))
        return np.array([s, BASE_DNA_RULE, d, 0, 1], dtype=np.int64)
    return chooser

print("Expected squared error per embedded random bit, (4^d - 1) / (6 d), and PSNR penalty versus 1-LSB:")
for d in (1, 2, 3):
    c = (4 ** d - 1) / (6 * d)
    print(f"  {d}-LSB: {c:.3f} per bit | ratio to 1-LSB {c/0.5:.2f} | {10*np.log10(c/0.5):.2f} dB")

rows = []
for s in (64, 128, 256):
    for i, p in enumerate(val_pairs[:2]):
        smp = prepare_sample(p, s)
        for name, ch in [("fixed 1-LSB", legacy_fixed_depth_chooser(1)), ("fixed 2-LSB", legacy_fixed_depth_chooser(2)),
                         ("fixed 3-LSB", legacy_fixed_depth_chooser(3)), ("random valid", make_random_valid_chooser(GLOBAL_SEED + i))]:
            env, _ = run_embedding_episode(smp, ch, use_reward=True)
            st = blocks_to_image(env.stego_blocks, smp["cover_image"].shape, BLOCK_SIZE)
            rows.append({"size": s, "policy": name, "v2 episode return": env.episode_return,
                         "PSNR": calculate_psnr(smp["cover_image"], st)})
print("\nv2 reward: episode return and PSNR by policy (does the v2 reward order the policies correctly?)")
print(pd.DataFrame(rows).groupby(["size", "policy"]).mean(numeric_only=True).round(4).to_string())

print("\nNormalised block features (q50 / q90 / q99 over the image), mean of 10 training covers:")
qs = np.mean([np.quantile(prepare_sample(p, 64)["state_matrix"], [0.5, 0.9, 0.99], axis=0) for p in train_pairs[:10]], axis=0)
print(pd.DataFrame(qs, index=["q50", "q90", "q99"], columns=["variance", "entropy", "edge_density", "glcm_contrast"]).round(3).to_string())

In [ ]:
# STEP 21S — Sequential block-wise environment. The agent sees the CURRENT block's features and decides
# [DNA rule, LSB depth, pixel location, embed/skip]. Reward = -(distortion relative to plain sequential LSB).
def split_seq_masks(masks):
    masks = np.asarray(masks)
    return [masks[SEQ_HEAD_OFFSETS[i]:SEQ_HEAD_OFFSETS[i + 1]] for i in range(len(SEQ_HEAD_SIZES))]

def block_se(a, b):
    d = a.astype(np.float64) - b.astype(np.float64)
    return float(np.sum(d * d))

def block_ssim_loss(a, b):
    if np.array_equal(a, b):
        return 0.0
    return 1.0 - float(np.clip(fast_block_ssim(a, b), 0.0, 1.0))

_ref_cache = {}
def plain_lsb_reference(sample):
    """Sum of squared error and of (1 - SSIM) of plain sequential LSB of the same payload (same depth rule as the LSB baseline)."""
    key = (sample["pair"]["pair_id"], sample["secret_size"])
    if key not in _ref_cache:
        bits = sample["payload_bits"]; total = len(bits)
        d_ref = max(1, math.ceil(total / (NUM_BLOCKS * EMBED_SPAN * 3)))
        m = math.ceil(total / (EMBED_SPAN * 3 * d_ref))
        se_sum = ssim_sum = 0.0; ptr = 0
        for b in range(m):
            blk = sample["blocks"][b]
            sb, n = embed_action_in_block(blk, bits[ptr:], BASE_DNA_RULE, d_ref, 0, pixel_locations)
            se_sum += block_se(blk, sb); ssim_sum += block_ssim_loss(blk, sb); ptr += n
        assert ptr == total
        _ref_cache[key] = (max(se_sum, 1e-9), max(ssim_sum, 1e-9), d_ref)
    return _ref_cache[key]

def build_seq_observation(state_matrix, img_mean, cursor, payload_ptr, total_bits):
    cur = state_matrix[cursor] if cursor < NUM_BLOCKS else np.zeros(NUM_FEATURES, dtype=np.float32)
    remaining = max(0, total_bits - payload_ptr)
    pressure = min(1.0, remaining / max(1, (NUM_BLOCKS - cursor) * MAX_BLOCK_BITS))     # capacity pressure incl. the current block
    extra = np.array([payload_ptr / total_bits, pressure, total_bits / THEORETICAL_MAX_CAPACITY_BITS, cursor / NUM_BLOCKS], dtype=np.float32)
    return np.clip(np.concatenate([cur, img_mean, extra]).astype(np.float32), 0.0, 1.0)

class SequentialDNAEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, pairs, secret_sizes, pixel_locations_, size_probs=None, use_reward=True, base_dna_rule=BASE_DNA_RULE):
        super().__init__()
        self.pairs, self.secret_sizes = list(pairs), list(secret_sizes)
        p = np.asarray(size_probs if size_probs is not None else np.ones(len(self.secret_sizes)), dtype=np.float64)
        self.size_probs = p / p.sum()
        self.pixel_locations, self.base_dna_rule, self.use_reward = pixel_locations_, base_dna_rule, use_reward
        self.observation_space = spaces.Box(0.0, 1.0, (SEQ_OBS_DIM,), dtype=np.float32)
        self.action_space = spaces.MultiDiscrete(SEQ_HEAD_SIZES)
        self.rng = np.random.RandomState(GLOBAL_SEED)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.RandomState(seed)
        if options and "sample" in options:
            sample = options["sample"]
        else:
            pair = self.pairs[int(self.rng.randint(len(self.pairs)))]
            size = int(self.rng.choice(self.secret_sizes, p=self.size_probs))
            sample = prepare_sample(pair, size)
        self.sample = sample
        self.cover_blocks, self.stego_blocks = sample["blocks"], sample["blocks"].copy()
        self.state_matrix, self.payload_bits = sample["state_matrix"], sample["payload_bits"]
        self.img_mean = self.state_matrix.mean(axis=0).astype(np.float32)
        self.total_bits, self.secret_size = len(self.payload_bits), sample["secret_size"]
        if self.use_reward:
            self.se_ref, self.ssim_ref, self.d_ref = plain_lsb_reference(sample)
        self.cursor, self.payload_ptr, self.invalid_picks, self.episode_return = 0, 0, 0, 0.0
        self.block_bit_log = {}
        return build_seq_observation(self.state_matrix, self.img_mean, self.cursor, self.payload_ptr, self.total_bits), {}

    def action_masks(self):
        """Skipping / shallow depths are forbidden as soon as they would make completing the payload impossible."""
        remaining = int(self.total_bits - self.payload_ptr)
        left_after = (NUM_BLOCKS - self.cursor - 1) * MAX_BLOCK_BITS
        dna = np.ones(NUM_DNA_RULES, dtype=bool); loc = np.ones(NUM_PIXEL_LOCATIONS, dtype=bool)
        lsb = np.ones(NUM_LSB_OPTIONS, dtype=bool); emb = np.ones(NUM_EMBED_FLAGS, dtype=bool)
        if remaining > 0:
            for k in range(NUM_LSB_OPTIONS):
                lsb[k] = (left_after + EMBED_SPAN * 3 * (k + 1)) >= remaining
            emb[0] = left_after >= remaining
            if not lsb.any():
                lsb[-1] = True
        return np.concatenate([dna, lsb, loc, emb])

    def step(self, action):
        dna_rule_id, lsb_amount = int(action[0]), int(action[1]) + 1
        location_id, embed_flag = int(action[2]), int(action[3])
        m = split_seq_masks(self.action_masks())
        if embed_flag == 0 and not m[3][0]:                  # unreachable under correct masking
            embed_flag = 1; self.invalid_picks += 1
        if embed_flag == 1 and not m[1][lsb_amount - 1]:     # unreachable under correct masking
            self.invalid_picks += 1
            ok = np.flatnonzero(m[1]); up = ok[ok >= lsb_amount - 1]
            lsb_amount = int((up[0] if len(up) else ok[-1]) + 1)
        b = self.cursor
        original = self.cover_blocks[b]
        remaining = self.payload_bits[self.payload_ptr:]
        reward, bits_used = 0.0, 0
        if embed_flag == 1 and len(remaining) > 0:
            stego_block, bits_used = embed_action_in_block(original, remaining, dna_rule_id, lsb_amount, location_id,
                                                           self.pixel_locations, self.base_dna_rule)
            self.stego_blocks[b] = stego_block
            self.block_bit_log[b] = {"embedded": True, "dna_rule": dna_rule_id, "lsb": lsb_amount, "location": location_id,
                                     "bits": int(bits_used), "payload_start": int(self.payload_ptr)}
            if self.use_reward:
                cost = (block_se(original, stego_block) / self.se_ref
                        + SSIM_COST_WEIGHT * block_ssim_loss(original, stego_block) / self.ssim_ref)
                reward = -float(min(cost, STEP_REWARD_CLIP))
        else:
            self.block_bit_log[b] = {"embedded": False, "dna_rule": None, "lsb": None, "location": None,
                                     "bits": 0, "payload_start": int(self.payload_ptr)}
        self.cursor += 1
        self.payload_ptr += bits_used
        completed = self.payload_ptr >= self.total_bits
        terminated = bool(completed or self.cursor >= NUM_BLOCKS)
        if self.use_reward and terminated and not completed:      # unreachable under the feasibility mask
            reward -= INCOMPLETE_PENALTY_SCALE * (self.total_bits - self.payload_ptr) / self.total_bits
        self.episode_return += reward
        obs = build_seq_observation(self.state_matrix, self.img_mean, self.cursor, self.payload_ptr, self.total_bits)
        info = {"bits_embedded": int(bits_used), "payload_ptr": int(self.payload_ptr), "block_id": int(b),
                "secret_size": self.secret_size, "completed": bool(completed)}
        return obs, float(reward), terminated, False, info

print("SequentialDNAEnv defined. Observation dimension:", SEQ_OBS_DIM, "| action heads:", SEQ_HEAD_SIZES)

In [ ]:
# STEP 22S — From here on the ACTIVE layout is the sequential environment. The legacy environment (STEP 21) must not be run after this step.
OBS_DIM = SEQ_OBS_DIM
HEAD_SIZES = list(SEQ_HEAD_SIZES)
HEAD_OFFSETS = SEQ_HEAD_OFFSETS
split_action_masks = split_seq_masks                  # make_random_valid_chooser (STEP 22) now follows the sequential layout

def run_embedding_episode(sample, chooser, use_reward=False):
    env = SequentialDNAEnv([sample["pair"]], [sample["secret_size"]], pixel_locations, use_reward=use_reward)
    obs, _ = env.reset(options={"sample": sample})
    done, steps = False, 0
    while not done:
        obs, _, terminated, truncated, _ = env.step(chooser(obs, env.action_masks()))
        done = terminated or truncated
        steps += 1
    return env, steps

def make_fixed_depth_chooser(depth):
    """Embeds in every block (raster order) with a fixed depth, raised only if the feasibility mask requires it."""
    def chooser(obs, masks):
        parts = split_action_masks(masks)
        ok, want = np.flatnonzero(parts[1]), depth - 1
        d = want if parts[1][want] else (int(ok[ok >= want][0]) if (ok >= want).any() else int(ok[-1]))
        return np.array([BASE_DNA_RULE, d, 0, 1], dtype=np.int64)
    return chooser

assert OBS_DIM == 12 and HEAD_SIZES == [8, 3, 64, 2]
print("Active layout: observation", OBS_DIM, "| heads", HEAD_SIZES)

In [ ]:
# STEP 23S — Environment validation for all sizes: exact extraction, reward bookkeeping, reward ordering
check_env(ActionMasker(SequentialDNAEnv(train_pairs[:3], [64], pixel_locations), _mask_fn), warn=True)
print("Gymnasium environment check passed.")
rows = []
for size in SECRET_SIZES:
    smp = prepare_sample(train_pairs[0], size)
    total = len(smp["payload_bits"])
    env, steps = run_embedding_episode(smp, make_random_valid_chooser(GLOBAL_SEED), use_reward=True)
    recs = records_from_log(env.block_bit_log)
    ok, msg = check_records(recs, total); assert ok, msg
    assert env.payload_ptr == total and env.invalid_picks == 0
    bits = np.concatenate([recover_bits_from_block(env.stego_blocks[r["block_index"]], r["dna_rule"], r["lsb_depth"],
                           r["pixel_location"], pixel_locations, r["bits_embedded"]) for r in recs])
    assert np.array_equal(bits, smp["secret_bits"]), "bit-exact extraction failed"
    rand_ret = env.episode_return
    d_ref = plain_lsb_reference(smp)[2]
    env_ref, _ = run_embedding_episode(smp, make_fixed_depth_chooser(d_ref), use_reward=True)
    assert abs(env_ref.episode_return + (1.0 + SSIM_COST_WEIGHT)) < 1e-6, "plain-LSB reference must score exactly -(1+lambda)"
    row = {"size": size, "reference depth": d_ref, "plain-LSB return": round(env_ref.episode_return, 6), "random-valid return": round(rand_ret, 3)}
    if size <= 256:
        e3, _ = run_embedding_episode(smp, make_fixed_depth_chooser(3), use_reward=True)
        assert e3.episode_return < env_ref.episode_return, "reward must prefer 1-LSB to 3-LSB when capacity allows it"
        row["fixed 3-LSB return"] = round(e3.episode_return, 3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("Validation passed: exact extraction, no invalid picks, reference return = -(1+lambda), 1-LSB preferred to 3-LSB.")

In [ ]:
# STEP 24 — Training environment (training pairs only) and validation specifications (validation pairs only)
assert all(p["split"] == "train" for p in train_pairs) and all(p["split"] == "val" for p in val_pairs)
_cov = {sp: {p["cover_id"] for p in ps} for sp, ps in (("train", train_pairs), ("val", val_pairs), ("test", test_pairs))}
_sec = {sp: {p["secret_id"] for p in ps} for sp, ps in (("train", train_pairs), ("val", val_pairs), ("test", test_pairs))}
for d in (_cov, _sec):
    assert not (d["train"] & d["val"]) and not (d["train"] & d["test"]) and not (d["val"] & d["test"]), "Leakage between partitions."

def pick_val_pairs(n):
    chosen, seen = [], set()
    for p in val_pairs:
        if p["cover_id"] not in seen:
            chosen.append(p); seen.add(p["cover_id"])
        if len(chosen) == n: break
    return chosen

base_training_env = SequentialDNAEnv(train_pairs, SECRET_SIZES, pixel_locations, size_probs=TRAIN_SIZE_PROBS)
training_env = ActionMasker(base_training_env, _mask_fn)
val_specs = [(p, s) for s in SECRET_SIZES for p in pick_val_pairs(VAL_PAIRS_PER_SIZE.get(s, 0))]
print(f"Training environment created with {len(train_pairs)} training pairs and sizes {SECRET_SIZES}.")
print(f"Validation episodes per stage: {len(val_specs)} (validation pairs only; the test set is never used for training or model selection)")
print("Observation space:", training_env.observation_space, "| Action space:", training_env.action_space)

In [ ]:
# STEP 25 — Create or restore the MaskablePPO model. The previous best model is loaded READ ONLY if it is compatible.
CKPT_PREFIX = f"ppo_{MODEL_VERSION}_ckpt"
MODEL_FINAL_PATH = os.path.join(MODEL_DIR, f"ppo_{MODEL_VERSION}_final")
MODEL_BEST_PATH = os.path.join(MODEL_DIR, f"ppo_{MODEL_VERSION}_best")
FINAL_ZIP, BEST_ZIP = MODEL_FINAL_PATH + ".zip", MODEL_BEST_PATH + ".zip"
TRAIN_EPISODE_LOG = os.path.join(LOG_DIR, "training_episodes.csv")
VAL_HISTORY_FILE = os.path.join(LOG_DIR, "validation_history.csv")
VAL_DETAIL_FILE = os.path.join(LOG_DIR, "validation_detail.csv")
WARM_INFO_FILE = os.path.join(MODEL_DIR, f"ppo_{MODEL_VERSION}_warm_start_info.json")
ALLOW_SCRATCH_FALLBACK = True
CUSTOM_OBJECTS = {"gae_lambda": PPO_PARAMS["gae_lambda"]}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def build_ppo(env_):
    p = PPO_PARAMS
    return MaskablePPO("MlpPolicy", env_, learning_rate=p["lr"], n_steps=p["n_steps"], batch_size=p["batch_size"],
                       n_epochs=p["n_epochs"], gamma=p["gamma"], gae_lambda=p["gae_lambda"], clip_range=p["clip_range"],
                       ent_coef=p["ent_coef"], vf_coef=p["vf_coef"], max_grad_norm=p["max_grad_norm"],
                       target_kl=p["target_kl"], policy_kwargs=dict(net_arch=dict(pi=p["net_arch_pi"], vf=p["net_arch_vf"])),
                       seed=GLOBAL_SEED, device="cpu", verbose=0)

def latest_checkpoint():
    files = glob.glob(os.path.join(CHECKPOINT_DIR, f"{CKPT_PREFIX}_*_steps.zip"))
    return max(files, key=lambda f: int(re.search(r"_(\d+)_steps", f).group(1))) if files else None

def check_model_compatible(m):
    assert tuple(m.observation_space.shape) == (OBS_DIM,), "Checkpoint observation space differs from this environment."
    assert [int(x) for x in m.action_space.nvec] == HEAD_SIZES, "Checkpoint action space differs from this environment."
    assert m.policy_kwargs.get("net_arch") == dict(pi=PPO_PARAMS["net_arch_pi"], vf=PPO_PARAMS["net_arch_vf"]), \
        "Checkpoint network architecture differs from this configuration."

WARM_INFO, model = {}, None
ckpt = latest_checkpoint() if RESUME_TRAINING else None
if ckpt:
    model = MaskablePPO.load(ckpt, env=training_env, device="cpu", custom_objects=CUSTOM_OBJECTS)
    check_model_compatible(model)
    if os.path.exists(WARM_INFO_FILE):
        with open(WARM_INFO_FILE) as f: WARM_INFO = json.load(f)
    print(f"Resuming from {os.path.basename(ckpt)} ({model.num_timesteps} timesteps). Compatibility verified.")
else:
    if os.path.exists(OLD_MODEL_ZIP):
        old_sha = sha256_file(OLD_MODEL_ZIP)
        try:
            cand = MaskablePPO.load(OLD_MODEL_ZIP, env=training_env, device="cpu", custom_objects=CUSTOM_OBJECTS)
            check_model_compatible(cand)
            model = cand
            WARM_INFO = {"warm_start": True, "source": os.path.abspath(OLD_MODEL_ZIP), "old_sha256": old_sha,
                         "warm_start_timesteps": int(model.num_timesteps)}
            model.save(os.path.join(CHECKPOINT_DIR, f"{CKPT_PREFIX}_{model.num_timesteps}_steps"))
            print(f"Previous best model loaded as the starting checkpoint ({model.num_timesteps} timesteps, SHA-256 {old_sha[:16]}...).")
            print("Observation space, action space and architecture are compatible. The original file is not modified.")
        except Exception as e:
            print("The previous model is NOT compatible:", e)
            if not ALLOW_SCRATCH_FALLBACK: raise
    else:
        print("Previous model not found at", OLD_MODEL_ZIP)
    if model is None:
        model = build_ppo(training_env)
        WARM_INFO = {"warm_start": False, "old_sha256": None, "warm_start_timesteps": 0}
        print("New MaskablePPO model created from scratch (0 timesteps).")
    with open(WARM_INFO_FILE, "w") as f:
        json.dump(WARM_INFO, f, indent=1)

WARM_STEPS = int(WARM_INFO.get("warm_start_timesteps", 0))
TARGET_TOTAL_STEPS = WARM_STEPS + (TRAIN_BUDGET_STEPS if WARM_INFO.get("warm_start") else SCRATCH_BUDGET_STEPS)
print(f"Policy parameters: {sum(p.numel() for p in model.policy.parameters())} | current timesteps {model.num_timesteps} | "
      f"training target {TARGET_TOTAL_STEPS} timesteps | GAE lambda {model.gae_lambda}")

In [ ]:
# STEP 26 — Validation evaluation (validation pairs only; the test set is never used here)
def evaluate_on_validation(chooser_factory, specs):
    rows = []
    for pair, s in specs:
        smp = prepare_sample(pair, s)
        t0 = time.perf_counter()
        env, steps = run_embedding_episode(smp, chooser_factory(), use_reward=True)
        el = time.perf_counter() - t0
        recs = records_from_log(env.block_bit_log)
        stego = blocks_to_image(env.stego_blocks, smp["cover_image"].shape, BLOCK_SIZE)
        depth = np.array([r["lsb_depth"] for r in recs]) if recs else np.zeros(1)
        rules = Counter(r["dna_rule"] for r in recs); n = max(1, len(recs))
        rows.append({"pair_id": pair["pair_id"], "size": s, "episode_return": float(env.episode_return),
                     "completed": bool(env.payload_ptr >= env.total_bits), "payload_fraction": env.payload_ptr / env.total_bits,
                     "blocks_used": len(recs), "blocks_skipped": steps - len(recs), "mean_lsb": float(depth.mean()),
                     "frac_lsb1": float((depth == 1).mean()), "frac_lsb2": float((depth == 2).mean()), "frac_lsb3": float((depth == 3).mean()),
                     **{f"dna{k}": rules.get(k, 0) / n for k in range(NUM_DNA_RULES)},
                     "mse": calculate_mse(smp["cover_image"], stego), "psnr": min(calculate_psnr(smp["cover_image"], stego), 100.0),
                     "ssim": calculate_ssim(smp["cover_image"], stego), "val_time_s": el})
    return pd.DataFrame(rows)

def summarize_validation(df):
    per = df.groupby("size").mean(numeric_only=True)          # every secret size has equal weight
    out = {"val_score": float(per["episode_return"].mean()), "completion_rate": float(df["completed"].mean())}
    for c in ["payload_fraction", "blocks_used", "blocks_skipped", "mean_lsb", "frac_lsb1", "frac_lsb2", "frac_lsb3",
              "mse", "psnr", "ssim"] + [f"dna{k}" for k in range(NUM_DNA_RULES)]:
        out[c] = float(per[c].mean())
    for s in per.index:
        out[f"return_s{s}"] = float(per.loc[s, "episode_return"]); out[f"psnr_s{s}"] = float(per.loc[s, "psnr"])
    out["val_time_s"] = float(df["val_time_s"].sum())
    return out

# One-time reference: a random valid policy on the same validation episodes (for judging whether PPO learns)
REF_FILE = os.path.join(LOG_DIR, "validation_reference_random.csv")
if not os.path.exists(REF_FILE):
    ref_df = evaluate_on_validation(lambda: make_random_valid_chooser(GLOBAL_SEED), val_specs)
    ref_df.to_csv(REF_FILE, index=False)
ref_sum = summarize_validation(pd.read_csv(REF_FILE))
print(f"Reference (random valid policy): score {ref_sum['val_score']:.4f} | PSNR {ref_sum['psnr']:.2f} dB | mean LSB {ref_sum['mean_lsb']:.2f}")

In [ ]:
# STEP 27 — Staged training from the warm start: train a stage, validate, continue only while validation improves
class EpisodeLogger(BaseCallback):
    def __init__(self):
        super().__init__(); self.rows = []
    def _on_step(self):
        for info in self.locals.get("infos", []):
            ep = info.get("episode")
            if ep is not None:
                self.rows.append({"timesteps": int(self.model.num_timesteps), "reward": float(ep["r"]), "length": int(ep["l"]),
                                  "secret_size": int(info.get("secret_size", 0)), "completed": bool(info.get("completed", False))})
        return True

def stage_row(summary, timesteps, train_reward, train_time, improved, stage):
    return {"stage": stage, "timesteps": int(timesteps), "train_ep_reward": train_reward, "train_time_s": train_time,
            "improved": bool(improved), **summary}

history = pd.read_csv(VAL_HISTORY_FILE).to_dict("records") if os.path.exists(VAL_HISTORY_FILE) else []
if not history:
    d0 = evaluate_on_validation(lambda: PPOChooser(model), val_specs)
    d0.assign(timesteps=model.num_timesteps).to_csv(VAL_DETAIL_FILE, index=False)
    s0 = summarize_validation(d0)
    history.append(stage_row(s0, model.num_timesteps, float("nan"), 0.0, True, "start"))
    model.save(MODEL_BEST_PATH)                      # the starting model is the first best-model candidate
    pd.DataFrame(history).to_csv(VAL_HISTORY_FILE, index=False)
    print(f"Starting model ({model.num_timesteps} timesteps): val score {s0['val_score']:.4f} | mean LSB {s0['mean_lsb']:.2f} | "
          f"PSNR {s0['psnr']:.2f} dB | random-valid reference {ref_sum['val_score']:.4f}")

best_score = max(h["val_score"] for h in history)
stale = 0
for h in reversed([h for h in history if h["stage"] == "train"]):
    if h["improved"]: break
    stale += 1
stop_reason = "training budget reached"

while model.num_timesteps < TARGET_TOTAL_STEPS:
    if stale >= PATIENCE:
        stop_reason = f"no validation improvement for {PATIENCE} stages"; break
    print(f"PPO training started: stage {model.num_timesteps} -> {model.num_timesteps + STAGE_STEPS} timesteps.")
    logger = EpisodeLogger()
    t0 = time.perf_counter()
    model.learn(total_timesteps=STAGE_STEPS, callback=logger, reset_num_timesteps=False)
    train_time = time.perf_counter() - t0
    model.save(os.path.join(CHECKPOINT_DIR, f"{CKPT_PREFIX}_{model.num_timesteps}_steps"))
    if logger.rows:
        old = pd.read_csv(TRAIN_EPISODE_LOG) if os.path.exists(TRAIN_EPISODE_LOG) else pd.DataFrame()
        pd.concat([old, pd.DataFrame(logger.rows)], ignore_index=True).to_csv(TRAIN_EPISODE_LOG, index=False)
    train_reward = float(np.mean([e["reward"] for e in logger.rows[-50:]])) if logger.rows else float("nan")

    dv = evaluate_on_validation(lambda: PPOChooser(model), val_specs)
    summ = summarize_validation(dv)
    improved = (summ["val_score"] > best_score + MIN_IMPROVEMENT) and summ["completion_rate"] == 1.0
    if improved:
        best_score, stale = summ["val_score"], 0
        model.save(MODEL_BEST_PATH)
    else:
        stale += 1
    history.append(stage_row(summ, model.num_timesteps, train_reward, train_time, improved, "train"))
    pd.DataFrame(history).to_csv(VAL_HISTORY_FILE, index=False)
    old = pd.read_csv(VAL_DETAIL_FILE) if os.path.exists(VAL_DETAIL_FILE) else pd.DataFrame()
    pd.concat([old, dv.assign(timesteps=model.num_timesteps)], ignore_index=True).to_csv(VAL_DETAIL_FILE, index=False)
    print(f"[Stage {model.num_timesteps}] train reward {train_reward:.4f} | val score {summ['val_score']:.4f} | "
          f"completion {summ['completion_rate']*100:.0f}% | blocks {summ['blocks_used']:.0f} | "
          f"LSB 1/2/3 = {summ['frac_lsb1']:.2f}/{summ['frac_lsb2']:.2f}/{summ['frac_lsb3']:.2f} (mean {summ['mean_lsb']:.2f}) | "
          f"MSE {summ['mse']:.3f} | PSNR {summ['psnr']:.2f} dB | SSIM {summ['ssim']:.4f} | train {train_time/60:.1f} min")
    print("  Validation improved. Training continues." if improved else f"  No improvement ({stale}/{PATIENCE}).")
    if summ["frac_lsb3"] > 0.95:
        print("  Note: more than 95% of blocks still use 3 LSB (no adaptive depth yet).")
print(f"PPO training finished: {model.num_timesteps} timesteps | {stop_reason}.")

In [ ]:
# STEP 28 — Save final and best models separately; the deployed model is the last model that was actually saved as best
model.save(MODEL_FINAL_PATH)
for pth in (FINAL_ZIP, BEST_ZIP):
    if os.path.exists(pth): assert_in_root(pth)
assert os.path.exists(BEST_ZIP), "No best model was saved."
DEPLOYED_MODEL_ZIP = BEST_ZIP

PROBE_SEED = 2024
def ppo_probe_action(model_):
    obs_ = np.random.RandomState(PROBE_SEED).rand(OBS_DIM).astype(np.float32)
    a, _ = model_.predict(obs_, action_masks=np.ones(sum(HEAD_SIZES), dtype=bool), deterministic=True)
    return [int(x) for x in a]

ppo_model = MaskablePPO.load(DEPLOYED_MODEL_ZIP, device="cpu")
check_model_compatible(ppo_model)
MODEL_SHA256 = sha256_file(DEPLOYED_MODEL_ZIP)
PPO_MODEL_INFO = {"version": MODEL_VERSION, "file": os.path.basename(DEPLOYED_MODEL_ZIP), "sha256": MODEL_SHA256,
                  "num_timesteps": int(ppo_model.num_timesteps), "obs_dim": int(OBS_DIM), "action_nvec": [int(x) for x in HEAD_SIZES],
                  "probe_seed": PROBE_SEED, "probe_action": ppo_probe_action(ppo_model),
                  "warm_start_from_sha256": WARM_INFO.get("old_sha256"), "environment": "sequential_v3"}
with open(os.path.join(MODEL_DIR, f"ppo_{MODEL_VERSION}_deployed_info.json"), "w") as f:
    json.dump(PPO_MODEL_INFO, f, indent=1)

hist = pd.read_csv(VAL_HISTORY_FILE)
print("Validation history:")
print(hist[["stage", "timesteps", "train_ep_reward", "val_score", "completion_rate", "mean_lsb", "frac_lsb1", "frac_lsb2", "frac_lsb3",
            "mse", "psnr", "ssim", "improved"]].round(4).to_string(index=False))
saved = hist[hist.improved == True].iloc[-1]
print(f"\nDeployed model = last saved best: stage '{saved.stage}', {int(saved.timesteps)} timesteps, validation score {saved.val_score:.4f}.")
print(f"Random-valid reference: score {ref_sum['val_score']:.4f} | PSNR {ref_sum['psnr']:.2f} dB | mean LSB {ref_sum['mean_lsb']:.2f}")
if saved.stage == "start":
    print("No training stage improved on the untrained model: training did not help.")
print("Deployed model file:", DEPLOYED_MODEL_ZIP, f"(SHA-256 {MODEL_SHA256[:16]}...)")
print("Final model:", FINAL_ZIP)

In [ ]:
# STEP 28B — Adaptivity validation on VALIDATION pairs, all four secret sizes
from scipy.stats import spearmanr
ADAPT_PAIRS = {64: 6, 128: 6, 256: 4, 512: 2}
FEATURE_NAMES = ["variance", "entropy", "edge_density", "glcm_contrast"]
PPO_NAME = "PPO (deployed, deterministic)"

class StochasticPPOChooser:
    def __init__(self, model_): self.model_ = model_
    def __call__(self, obs, masks):
        return self.model_.predict(obs, action_masks=masks, deterministic=False)[0]

def adaptivity_validation(model_):
    makers = {PPO_NAME: lambda i: make_ppo_chooser(model_),
              "PPO (stochastic, diagnostic)": lambda i: StochasticPPOChooser(model_),
              "Random valid": lambda i: make_random_valid_chooser(GLOBAL_SEED + i),
              "Fixed 1-LSB (raised only if forced)": lambda i: make_fixed_depth_chooser(1),
              "Fixed 3-LSB": lambda i: make_fixed_depth_chooser(3)}
    summary, blocks = [], []
    for s, n in ADAPT_PAIRS.items():
        for i, p in enumerate(pick_val_pairs(n)):
            smp = prepare_sample(p, s); st = smp["state_matrix"]; total = len(smp["payload_bits"])
            for name, mk in makers.items():
                env, steps = run_embedding_episode(smp, mk(i), use_reward=True)
                recs = records_from_log(env.block_bit_log)
                stego = blocks_to_image(env.stego_blocks, smp["cover_image"].shape, BLOCK_SIZE)
                dep = np.array([r["lsb_depth"] for r in recs]) if recs else np.zeros(1)
                summary.append({"size": s, "policy": name, "completed": bool(env.payload_ptr >= total), "bits": int(env.payload_ptr),
                                "blocks_used": len(recs), "blocks_visited": steps, "lsb1": float((dep == 1).mean()),
                                "lsb2": float((dep == 2).mean()), "lsb3": float((dep == 3).mean()), "mean_lsb": float(dep.mean()),
                                "return": float(env.episode_return), "mse": calculate_mse(smp["cover_image"], stego),
                                "psnr": calculate_psnr(smp["cover_image"], stego), "ssim": calculate_ssim(smp["cover_image"], stego)})
                if name == PPO_NAME:
                    for b, l in env.block_bit_log.items():
                        blocks.append({"size": s, "pair": p["pair_id"], "block": int(b), "embedded": bool(l["embedded"]),
                                       "depth": int(l["lsb"]) if l["embedded"] else 0,
                                       "pressure": min(1.0, (total - l["payload_start"]) / ((NUM_BLOCKS - b) * MAX_BLOCK_BITS)),
                                       **{f: float(st[b, j]) for j, f in enumerate(FEATURE_NAMES)}})
    return pd.DataFrame(summary), pd.DataFrame(blocks)

def feature_dependence(bdf):
    rows = []
    for s, g in bdf.groupby("size"):
        e, skipped = g[g.embedded], g[~g.embedded]
        for f in FEATURE_NAMES:
            row = {"size": s, "feature": f, "embedded blocks": len(e), "depth std": round(float(e.depth.std()), 3) if len(e) > 1 else 0.0,
                   "mean feature, embedded": round(float(e[f].mean()), 4),
                   "mean feature, skipped": round(float(skipped[f].mean()), 4) if len(skipped) else float("nan")}
            if len(e) > 30 and e.depth.nunique() > 1 and e[f].nunique() > 1:
                rho, pval = spearmanr(e[f], e.depth)
                rs, ws = [], []
                for _, h in e.groupby(pd.qcut(e.pressure, 5, duplicates="drop"), observed=True):
                    if len(h) >= 30 and h.depth.nunique() > 1 and h[f].nunique() > 1:
                        rs.append(spearmanr(h[f], h.depth)[0]); ws.append(len(h))
                q = e.groupby(pd.qcut(e[f], 4, duplicates="drop"), observed=True).depth.mean().round(2).tolist()
                row.update({"Spearman(depth, feature)": round(float(rho), 3), "p-value": float(pval),
                            "Spearman within pressure strata": round(float(np.average(rs, weights=ws)), 3) if rs else float("nan"),
                            "mean depth by feature quartile": q})
            else:
                row["note"] = "depth constant (or too few blocks): no depth-feature relation to measure"
            rows.append(row)
    return pd.DataFrame(rows)

sdf, bdf = adaptivity_validation(ppo_model)
agg = sdf.groupby(["size", "policy"]).agg(completed=("completed", "mean"), bits=("bits", "mean"), blocks_used=("blocks_used", "mean"),
                                          visited=("blocks_visited", "mean"), lsb1=("lsb1", "mean"), lsb2=("lsb2", "mean"),
                                          lsb3=("lsb3", "mean"), mean_lsb=("mean_lsb", "mean"), ret=("return", "mean"),
                                          mse=("mse", "mean"), psnr=("psnr", "mean"), ssim=("ssim", "mean")).round(4)
print(agg.to_string())
agg.to_csv(os.path.join(LOG_DIR, "adaptivity_policies_validation.csv"))
fd = feature_dependence(bdf)
print("\nFeature dependence of the deployed PPO (embedded blocks; skipped vs embedded block features):")
print(fd.to_string(index=False))
fd.to_csv(os.path.join(LOG_DIR, "adaptivity_feature_dependence_validation.csv"), index=False)

ppo_rows = sdf[sdf.policy == PPO_NAME]
print("\nDeployed PPO depth distribution by size (fraction of embedded blocks):")
print(ppo_rows.groupby("size")[["lsb1", "lsb2", "lsb3", "mean_lsb"]].mean().round(3).to_string())
print("All episodes complete the payload:", bool(ppo_rows.completed.all()))
varying = fd[(fd.get("depth std", 0) > 0) & fd["Spearman within pressure strata"].notna()] if "Spearman within pressure strata" in fd else fd.iloc[0:0]
dep_feat = bool(len(varying) and (varying["Spearman within pressure strata"].abs() >= 0.2).any())
sel_feat = bool(((fd["mean feature, embedded"] - fd["mean feature, skipped"]).abs() >= 0.05).any())
print(f"Heuristic flags (diagnostic thresholds, not statistical tests): depth depends on a feature within pressure strata: {dep_feat} | "
      f"embedded and skipped blocks differ in a feature: {sel_feat}")

In [ ]:
# STEP 29 — Result record, output paths and the proposed-method sender
ALGORITHMS = ["Proposed_DNA_PPO", "LSB_Replacement", "LSB_Matching", "PVD"]
ALGO_LABELS = {"Proposed_DNA_PPO": "Proposed DNA + PPO", "LSB_Replacement": "LSB Replacement",
               "LSB_Matching": "LSB Matching", "PVD": "PVD"}
RESULT_COLUMNS = ["method", "pair_id", "cover_id", "secret_id", "split", "secret_size", "cover_size", "status", "note",
                  "payload_bits", "payload_rate_bpp", "payload_capacity_bits", "bits_embedded", "mse", "psnr", "ssim",
                  "embedding_time_s", "feature_time_s", "ber", "bit_accuracy", "exact_recovery", "extraction_completed",
                  "secret_mse", "secret_psnr", "secret_ssim", "secret_pixel_accuracy", "extraction_time_s",
                  "blocks_used", "blocks_skipped", "lsb_usage", "dna_rule_usage", "stego_path", "metadata_path", "recovered_path"]

def new_result(method, pair, size):
    r = {c: float("nan") for c in RESULT_COLUMNS}
    r.update(method=method, pair_id=pair["pair_id"], cover_id=pair["cover_id"], secret_id=pair["secret_id"], split=pair["split"],
             secret_size=int(size), cover_size=COVER_SIZE, status="NOT_RUN", note="", payload_bits=expected_payload_bits(size),
             payload_rate_bpp=expected_payload_bits(size) / (COVER_SIZE * COVER_SIZE), bits_embedded=0, extraction_completed=False,
             stego_path=None, metadata_path=None, recovered_path=None, lsb_usage=None, dna_rule_usage=None)
    return r

def out_path(base_dir, method, size, pair_id, suffix):
    d = os.path.join(base_dir, method, f"s{size}")
    os.makedirs(d, exist_ok=True)
    p = os.path.join(d, f"{pair_id}_{suffix}")
    assert_in_root(p)
    return p

def store_outputs(method, pair_id, size, stego, meta):
    sp, mp = out_path(STEGO_DIR, method, size, pair_id, "stego.png"), out_path(META_DIR, method, size, pair_id, "meta.json")
    Image.fromarray(stego).save(sp)
    assert np.array_equal(np.array(Image.open(sp).convert("RGB"), dtype=np.uint8), stego), "The saved PNG is not lossless."
    with open(mp, "w", encoding="utf-8") as f:
        json.dump(meta, f)
    return sp, mp

def payload_crc(bits):
    return int(zlib.crc32(np.packbits(bits).tobytes()) & 0xFFFFFFFF)

def build_metadata_proposed(sample, records, model_info):
    H, W, C = sample["cover_image"].shape
    n = sample["secret_size"]; payload = sample["payload_bits"]
    return {"metadata_version": METADATA_VERSION, "method": "Proposed_DNA_PPO", "pair_id": sample["pair"]["pair_id"],
            "cover_height": int(H), "cover_width": int(W), "cover_channels": int(C),
            "secret_height": int(n), "secret_width": int(n), "secret_channels": 1, "secret_bit_depth": 8,
            "payload_bit_order": "msb_first", "secret_scan_order": "row-major",
            "total_payload_bits": int(len(payload)), "payload_crc32": payload_crc(payload),
            "preprocessing_transform": "none", "block_size": BLOCK_SIZE, "embed_span": EMBED_SPAN, "max_lsb": MAX_LSB,
            "pixel_location_order": "row-major", "base_dna_rule": int(BASE_DNA_RULE),
            "dna_rules": [[rule[v] for v in range(4)] for rule in dna_mapping_rules],
            "ppo_model": model_info, "num_records": len(records),
            "embedded_bits_total": int(sum(r["bits_embedded"] for r in records)), "records": records}

def run_sender_proposed(pair, size, chooser, model_info):
    """Every decision comes from the PPO chooser. Returns (result, stego_image, metadata)."""
    sample = prepare_sample(pair, size)
    total = len(sample["payload_bits"])
    res = new_result("Proposed_DNA_PPO", pair, size)
    res["feature_time_s"] = sample["feat_time"]
    res["payload_capacity_bits"] = THEORETICAL_MAX_CAPACITY_BITS
    if total > THEORETICAL_MAX_CAPACITY_BITS:
        res.update(status="FAILED_CAPACITY", note=f"required {total} bits > available {THEORETICAL_MAX_CAPACITY_BITS} bits")
        return res, None, None
    t0 = time.perf_counter()
    env, steps = run_embedding_episode(sample, chooser)
    stego = blocks_to_image(env.stego_blocks, sample["cover_image"].shape, BLOCK_SIZE)
    res["embedding_time_s"] = time.perf_counter() - t0 + sample["feat_time"]
    log = env.block_bit_log
    records = records_from_log(log)
    res.update(bits_embedded=int(env.payload_ptr), blocks_used=len(records), blocks_skipped=steps - len(records),
               lsb_usage=dict(Counter(r["lsb_depth"] for r in records)), dna_rule_usage=dict(Counter(r["dna_rule"] for r in records)))
    if env.payload_ptr < total:
        res.update(status="FAILED_INCOMPLETE", note=f"embedded {env.payload_ptr}/{total} bits; nothing saved, nothing padded")
        return res, stego, None
    ok, msg = check_records(records, total)
    if not ok:
        res.update(status="FAILED_SYNC", note=msg)
        return res, stego, None
    res["status"] = "SENT"
    return res, stego, build_metadata_proposed(sample, records, model_info)

print("Proposed-method sender defined.")

In [ ]:
# STEP 30 — Receiver module (written to the new directory). Metadata structure is unchanged; validation is stricter.
RECEIVER_SOURCE = r'''
# receiver_final_ppo_v2.py - standalone receiver for the DNA + PPO steganography method
# Usage: python receiver_final_ppo_v2.py <stego.png> <metadata.json> <recovered.png> <ppo_model.zip>
import os, sys, json, time, zlib, hashlib
import numpy as np
from PIL import Image

class ReceiverError(Exception):
    pass

SUPPORTED_VERSIONS = ("1.0",)
REQUIRED_KEYS = ("metadata_version", "cover_height", "cover_width", "cover_channels", "secret_height", "secret_width",
                 "secret_channels", "total_payload_bits", "block_size", "embed_span", "max_lsb", "base_dna_rule",
                 "dna_rules", "records", "pixel_location_order")
INT_KEYS = ("cover_height", "cover_width", "cover_channels", "secret_height", "secret_width", "secret_channels",
            "total_payload_bits", "block_size", "embed_span", "max_lsb", "base_dna_rule")
RECORD_KEYS = ("block_index", "dna_rule", "lsb_depth", "pixel_location", "bits_embedded", "payload_start")
_MODEL_CACHE = {}

def _is_int(v):
    return isinstance(v, int) and not isinstance(v, bool)

def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def _image_to_blocks(image, bs):
    H, W, C = image.shape
    return image.reshape(H // bs, bs, W // bs, bs, C).swapaxes(1, 2).reshape(-1, bs, bs, C)

def _build_lut(rules):
    if not isinstance(rules, list) or len(rules) == 0:
        raise ReceiverError("The DNA rule table in the metadata is invalid.")
    for r in rules:
        if not isinstance(r, list) or len(r) != 4 or len(set(r)) != 4:
            raise ReceiverError("A DNA rule in the metadata is not a one-to-one mapping of 4 symbols.")
    rev = [{letter: v for v, letter in enumerate(r)} for r in rules]
    lut = np.zeros((len(rules), len(rules), 4), dtype=np.uint8)
    for s in range(len(rules)):
        for d in range(len(rules)):
            for v in range(4):
                if rules[s][v] not in rev[d]:
                    raise ReceiverError("DNA rules use different alphabets.")
                lut[s, d, v] = rev[d][rules[s][v]]
    return lut

def _apply_transform(bits, src, dst, lut):
    bits = np.asarray(bits, dtype=np.uint8).copy()
    n = len(bits) // 2
    if n > 0:
        pairs = bits[:2 * n].reshape(-1, 2)
        values = (pairs[:, 0] << 1) | pairs[:, 1]
        nv = lut[src, dst][values]
        bits[0:2 * n:2] = (nv >> 1) & 1
        bits[1:2 * n:2] = nv & 1
    return bits

def _extract_raw(block, depth, loc, n_bits, bs, span):
    n_slots = -(-n_bits // depth)
    if n_slots > span * 3:
        raise ReceiverError("A record asks for more bits than one block can hold.")
    s = np.arange(n_slots)
    pix = (loc + s // 3) % (bs * bs)
    vals = block[pix // bs, pix % bs, s % 3].astype(np.int64) & ((1 << depth) - 1)
    shifts = np.arange(depth - 1, -1, -1)                       # most significant embedded bit first
    return (((vals[:, None] >> shifts) & 1).reshape(-1)[:n_bits]).astype(np.uint8)

def _validate_metadata(meta):
    if not isinstance(meta, dict):
        raise ReceiverError("The metadata is not a JSON object.")
    missing = [k for k in REQUIRED_KEYS if k not in meta]
    if missing:
        raise ReceiverError("The metadata is missing required fields: %s." % ", ".join(missing))
    if meta["metadata_version"] not in SUPPORTED_VERSIONS:
        raise ReceiverError("Unsupported metadata version: %s." % meta["metadata_version"])
    for k in INT_KEYS:
        if not _is_int(meta[k]) or meta[k] < 0:
            raise ReceiverError("The metadata field '%s' must be a non-negative integer." % k)
    if meta["pixel_location_order"] != "row-major":
        raise ReceiverError("The pixel location order in the metadata is not supported.")
    if meta.get("payload_bit_order", "msb_first") != "msb_first" or meta.get("secret_scan_order", "row-major") != "row-major":
        raise ReceiverError("The payload bit order or secret scan order in the metadata is not supported.")
    if meta["block_size"] < 1 or meta["cover_height"] % meta["block_size"] or meta["cover_width"] % meta["block_size"]:
        raise ReceiverError("The cover dimensions are not a multiple of the block size.")
    if not (0 <= meta["base_dna_rule"] < len(meta["dna_rules"])):
        raise ReceiverError("The base DNA rule index is out of range.")
    if meta["secret_height"] * meta["secret_width"] * 8 != meta["total_payload_bits"] or meta["secret_channels"] != 1:
        raise ReceiverError("The secret image dimensions are inconsistent with the payload size.")
    if not isinstance(meta["records"], list):
        raise ReceiverError("The embedding records are invalid.")
    for r in meta["records"]:
        if not isinstance(r, dict) or any(k not in r or not _is_int(r[k]) for k in RECORD_KEYS):
            raise ReceiverError("An embedding record is incomplete or contains non-integer values.")
        if r["bits_embedded"] <= 0 or r["payload_start"] < 0:
            raise ReceiverError("An embedding record has an invalid length or payload position.")

def verify_ppo_model(meta, model_path):
    info = meta.get("ppo_model")
    if model_path is None or info is None:
        return {"checked": False}
    if not os.path.exists(model_path):
        raise ReceiverError("The PPO model file was not found: %s" % model_path)
    key = os.path.abspath(model_path)
    hit = _MODEL_CACHE.get(key)
    if hit is None:
        sha = _sha256_file(model_path)
        from sb3_contrib import MaskablePPO
        hit = {"sha": sha, "model": MaskablePPO.load(model_path, device="cpu")}
        _MODEL_CACHE[key] = hit
    if hit["sha"] != info["sha256"]:
        raise ReceiverError("PPO model mismatch: the model file differs from the one used by the sender (SHA-256).")
    model = hit["model"]
    if tuple(model.observation_space.shape) != (int(info["obs_dim"]),):
        raise ReceiverError("PPO model mismatch: observation dimension differs from the sender configuration.")
    if [int(x) for x in model.action_space.nvec] != [int(x) for x in info["action_nvec"]]:
        raise ReceiverError("PPO model mismatch: action space differs from the sender configuration.")
    obs = np.random.RandomState(int(info["probe_seed"])).rand(int(info["obs_dim"])).astype(np.float32)
    masks = np.ones(int(sum(info["action_nvec"])), dtype=bool)
    action, _ = model.predict(obs, action_masks=masks, deterministic=True)
    if [int(x) for x in action] != [int(x) for x in info["probe_action"]]:
        raise ReceiverError("PPO model mismatch: the probe prediction differs from the sender's.")
    return {"checked": True, "sha256": hit["sha"], "num_timesteps": int(model.num_timesteps), "version": info.get("version")}

def receiver_extract(stego, metadata, model_path=None, strict_crc=True, verify_model=True):
    """stego: PNG path or HxWx3 uint8 array. metadata: JSON path or dict.
    The sender's embedding records are followed exactly; no new embedding decisions are made.
    Noise never makes extraction raise: bit errors are reported through the recovered payload (strict_crc=False)."""
    t0 = time.perf_counter()
    if isinstance(metadata, (str, os.PathLike)):
        with open(metadata, "r", encoding="utf-8") as f:
            metadata = json.load(f)
    if isinstance(stego, (str, os.PathLike)):
        stego = np.array(Image.open(stego).convert("RGB"), dtype=np.uint8)
    _validate_metadata(metadata)
    if not isinstance(stego, np.ndarray) or stego.ndim != 3 or stego.dtype != np.uint8:
        raise ReceiverError("The stego image must be an HxWx3 uint8 array.")
    model_check = verify_ppo_model(metadata, model_path) if verify_model else {"checked": False}
    ch, cw, cc = metadata["cover_height"], metadata["cover_width"], metadata["cover_channels"]
    if stego.shape != (ch, cw, cc):
        raise ReceiverError("The stego image shape %s does not match the metadata %s." % (stego.shape, (ch, cw, cc)))
    bs, span, max_lsb = metadata["block_size"], metadata["embed_span"], metadata["max_lsb"]
    base, total = metadata["base_dna_rule"], metadata["total_payload_bits"]
    lut = _build_lut(metadata["dna_rules"])
    blocks = _image_to_blocks(stego, bs)
    records = sorted(metadata["records"], key=lambda r: r["payload_start"])
    expected, seen, pieces = 0, set(), []
    for r in records:
        if r["block_index"] in seen or not (0 <= r["block_index"] < len(blocks)):
            raise ReceiverError("Invalid or duplicated block index: %s." % r["block_index"])
        seen.add(r["block_index"])
        if r["payload_start"] != expected:
            raise ReceiverError("Gap or overlap in the payload at bit %d." % expected)
        if not (1 <= r["lsb_depth"] <= max_lsb) or not (0 <= r["pixel_location"] < bs * bs) or not (0 <= r["dna_rule"] < len(lut)):
            raise ReceiverError("A record contains an invalid LSB depth, pixel location or DNA rule.")
        raw = _extract_raw(blocks[r["block_index"]], r["lsb_depth"], r["pixel_location"], r["bits_embedded"], bs, span)
        pieces.append(_apply_transform(raw, r["dna_rule"], base, lut))
        expected += r["bits_embedded"]
    if expected != total:
        raise ReceiverError("The records cover %d bits, but the payload is %d bits (incomplete metadata)." % (expected, total))
    payload = np.concatenate(pieces) if pieces else np.zeros(0, dtype=np.uint8)
    if len(payload) != total:
        raise ReceiverError("The extracted payload length does not match the expected length.")
    crc_ok = None
    if "payload_crc32" in metadata:
        crc_ok = (zlib.crc32(np.packbits(payload).tobytes()) & 0xFFFFFFFF) == metadata["payload_crc32"]
        if not crc_ok and strict_crc:
            raise ReceiverError("CRC32 mismatch: the recovered payload is corrupted or the metadata is incorrect.")
    secret = np.packbits(payload).reshape(metadata["secret_height"], metadata["secret_width"]).astype(np.uint8)
    return {"secret_image": secret, "payload_bits": payload, "num_blocks_used": len(records),
            "crc_ok": crc_ok, "model_check": model_check, "elapsed_s": time.perf_counter() - t0}

def main(argv):
    if len(argv) != 5:
        print("Usage: python receiver_final_ppo_v2.py <stego.png> <metadata.json> <recovered.png> <ppo_model.zip>")
        return 2
    out = receiver_extract(argv[1], argv[2], model_path=argv[4])
    Image.fromarray(out["secret_image"]).save(argv[3])
    print(json.dumps({"status": "OK", "shape": list(out["secret_image"].shape), "blocks_used": out["num_blocks_used"],
                      "crc_ok": out["crc_ok"], "model_check": out["model_check"]}))
    return 0

if __name__ == "__main__":
    try:
        sys.exit(main(sys.argv))
    except ReceiverError as e:
        print("Receiver failed:", e)
        sys.exit(1)
'''
assert_in_root(RECEIVER_FILE)
with open(RECEIVER_FILE, "w", encoding="utf-8") as f:
    f.write(RECEIVER_SOURCE)
if _new_root not in sys.path:
    sys.path.insert(0, _new_root)
sys.modules.pop("receiver_final_ppo_v2", None)
import receiver_final_ppo_v2
from receiver_final_ppo_v2 import receiver_extract, ReceiverError
assert os.path.abspath(receiver_final_ppo_v2.__file__).startswith(_new_root + os.sep), "Receiver imported from the wrong folder."
print("Receiver module written to", RECEIVER_FILE)

In [ ]:
# STEP 31 — One complete cycle for any algorithm, then the proposed method with an independent receiver test
class CapacityError(Exception):
    pass

EXTRACTORS, BASELINE_EMBED = {}, {}      # filled here and in STEPS 32-34

def proposed_extract(stego, meta):
    return receiver_extract(stego, meta, model_path=DEPLOYED_MODEL_ZIP, strict_crc=False)["payload_bits"]
EXTRACTORS["Proposed_DNA_PPO"] = proposed_extract

def extract_with_method(method, stego, meta):
    return EXTRACTORS[method](stego, meta)

def pvd_capacity_bits(cover):
    p1, p2 = pvd_split_pairs(cover)
    idx = np.searchsorted(PVD_LOWER, np.abs(p1 - p2), side="right") - 1
    return int(PVD_NBITS[idx].sum())

BASELINE_CAPACITY = {"LSB_Replacement": lambda c: int(c.size * MAX_LSB), "LSB_Matching": lambda c: int(c.size),
                     "PVD": pvd_capacity_bits}

def fill_metrics(res, cover, stego, secret_img, secret_bits, rec_bits):
    size = res["secret_size"]
    assert len(rec_bits) == len(secret_bits), "Extracted payload length differs from the original payload."
    rec_img = np.packbits(rec_bits).reshape(size, size).astype(np.uint8)
    assert rec_img.shape == secret_img.shape
    m = bit_metrics(secret_bits, rec_bits)
    res.update(mse=calculate_mse(cover, stego), psnr=calculate_psnr(cover, stego), ssim=calculate_ssim(cover, stego),
               ber=m["ber"], bit_accuracy=m["bit_accuracy"], exact_recovery=m["exact_recovery"],
               extraction_completed=True, **secret_quality(secret_img, rec_img))
    res["status"] = "PASS" if (m["exact_recovery"] and np.array_equal(rec_img, secret_img)) else "FAILED_MISMATCH"
    return rec_img

def run_baseline_sender(method, pair, size, cover, bits):
    res = new_result(method, pair, size)
    res["payload_capacity_bits"] = BASELINE_CAPACITY[method](cover)
    t0 = time.perf_counter()
    try:
        stego, params = BASELINE_EMBED[method](cover, bits)
    except CapacityError as e:
        res.update(status="FAILED_CAPACITY", note=str(e))
        return res, None, None
    res["embedding_time_s"] = time.perf_counter() - t0
    res["bits_embedded"] = int(len(bits))
    meta = {"metadata_version": METADATA_VERSION, "method": method, "pair_id": pair["pair_id"],
            "cover_height": COVER_SIZE, "cover_width": COVER_SIZE, "cover_channels": 3,
            "secret_height": int(size), "secret_width": int(size), "total_payload_bits": int(len(bits)),
            "payload_crc32": payload_crc(bits), **params}
    res["status"] = "SENT"
    return res, stego, meta

def process_pair(method, pair, size):
    """Embed -> save stego PNG + JSON -> reload both from disk -> extract -> compare with the original secret."""
    try:
        cover, secret_img, bits = load_pair_arrays(pair, size)
        if method == "Proposed_DNA_PPO":
            res, stego, meta = run_sender_proposed(pair, size, make_ppo_chooser(ppo_model), PPO_MODEL_INFO)
        else:
            res, stego, meta = run_baseline_sender(method, pair, size, cover, bits)
        if res["status"] != "SENT":
            return res
        res["stego_path"], res["metadata_path"] = store_outputs(method, pair["pair_id"], size, stego, meta)
        stego_disk = np.array(Image.open(res["stego_path"]).convert("RGB"), dtype=np.uint8)
        with open(res["metadata_path"], "r", encoding="utf-8") as f:
            meta_disk = json.load(f)
        t0 = time.perf_counter()
        try:
            rec_bits = extract_with_method(method, stego_disk, meta_disk)
        except ReceiverError as e:
            res.update(status="FAILED_RECEIVER", note=str(e)); return res
        res["extraction_time_s"] = time.perf_counter() - t0
        rec_img = fill_metrics(res, cover, stego_disk, secret_img, bits, rec_bits)
        res["recovered_path"] = out_path(RECOVERED_DIR, method, size, pair["pair_id"], "recovered.png")
        Image.fromarray(rec_img).save(res["recovered_path"])
        return res
    except Exception as e:
        r = new_result(method, pair, size); r.update(status="ERROR", note=repr(e)); return r

# ---- proposed method: warm-up, then an independent receiver test in a fresh process ----
process_pair("Proposed_DNA_PPO", test_pairs[0], 64)
IND_PAIR, IND_SIZE = test_pairs[0], 64
res_i = process_pair("Proposed_DNA_PPO", IND_PAIR, IND_SIZE)
assert res_i["status"] == "PASS", res_i
print(f"Proposed method, pair {IND_PAIR['pair_id']} ({IND_SIZE}x{IND_SIZE}): {res_i['status']} | "
      f"MSE {res_i['mse']:.4f} | PSNR {res_i['psnr']:.3f} dB | SSIM {res_i['ssim']:.4f} | BER {res_i['ber']:.6f} | "
      f"embedding {res_i['embedding_time_s']:.2f} s | extraction {res_i['extraction_time_s']:.3f} s")

rec_path = out_path(RECOVERED_DIR, "Proposed_DNA_PPO", IND_SIZE, "independent_test", "recovered.png")
proc = subprocess.run([sys.executable, RECEIVER_FILE, res_i["stego_path"], res_i["metadata_path"], rec_path, DEPLOYED_MODEL_ZIP],
                      capture_output=True, text=True)
assert proc.returncode == 0, f"The receiver failed in a fresh process: {proc.stdout} {proc.stderr[-500:]}"
child = json.loads([ln for ln in proc.stdout.strip().splitlines() if ln.startswith("{")][-1])
truth = load_secret_image(IND_PAIR[f"secret_s{IND_SIZE}"], IND_SIZE)
recovered = np.array(Image.open(rec_path))
assert recovered.shape == truth.shape and np.array_equal(recovered, truth) and child["crc_ok"] is True
print("Independent receiver (fresh process): secret image recovered successfully.")

with open(res_i["metadata_path"]) as f: m0 = json.load(f)
def _drop(m): m.update(records=m["records"][:-1])
def _rule(m): m["records"][0].update(dna_rule=(m["records"][0]["dna_rule"] + 1) % 8)
def _model(m): m["ppo_model"]["sha256"] = "0" * 64
def _missing(m): m.pop("total_payload_bits")
def _badtype(m): m["records"][0]["lsb_depth"] = "3"
for name, tamper in [("dropped record", _drop), ("wrong DNA rule", _rule), ("different PPO model", _model),
                     ("missing field", _missing), ("wrong field type", _badtype)]:
    mm = json.loads(json.dumps(m0)); tamper(mm)
    try:
        receiver_extract(res_i["stego_path"], mm, model_path=DEPLOYED_MODEL_ZIP)
        raise AssertionError(f"The receiver accepted modified metadata: {name}.")
    except ReceiverError as e:
        print(f"Modified metadata rejected ({name}).")
print("Receiver validation completed.")

In [ ]:
# STEP 32 — LSB replacement: sequential, 1 LSB per sample wherever the payload fits (3 LSB for 512x512 payloads)
def lsb_replacement_embed(cover, bits):
    total = len(bits)
    depth = max(1, math.ceil(total / cover.size))
    if depth > MAX_LSB:
        raise CapacityError(f"payload {total} bits exceeds the {MAX_LSB}-LSB capacity {cover.size * MAX_LSB} bits")
    n_slots = -(-total // depth)
    padded = np.zeros(n_slots * depth, dtype=np.int64); padded[:total] = bits
    vals = padded.reshape(n_slots, depth) @ (1 << np.arange(depth - 1, -1, -1))
    flat = cover.reshape(-1).copy()
    keep = 0xFF ^ ((1 << depth) - 1)
    flat[:n_slots] = ((flat[:n_slots].astype(np.int64) & keep) | vals).astype(np.uint8)
    return flat.reshape(cover.shape), {"depth": int(depth)}

def lsb_replacement_extract(stego, meta):
    depth, total = int(meta["depth"]), int(meta["total_payload_bits"])
    n_slots = -(-total // depth)
    vals = stego.reshape(-1)[:n_slots].astype(np.int64) & ((1 << depth) - 1)
    return ((vals[:, None] >> np.arange(depth - 1, -1, -1)) & 1).reshape(-1)[:total].astype(np.uint8)

BASELINE_EMBED["LSB_Replacement"] = lsb_replacement_embed
EXTRACTORS["LSB_Replacement"] = lsb_replacement_extract
_r = process_pair("LSB_Replacement", test_pairs[0], 64)
assert _r["status"] == "PASS", _r
print(f"LSB Replacement self-test (64x64): {_r['status']} | PSNR {_r['psnr']:.3f} dB | BER {_r['ber']:.6f}")

In [ ]:
# STEP 33 — LSB matching: if the LSB differs from the payload bit, the sample changes by +1 or -1
def lsb_matching_embed(cover, bits):
    total = len(bits)
    if total > cover.size:
        raise CapacityError(f"payload {total} bits exceeds the 1 bit per sample capacity {cover.size} bits")
    rng = np.random.RandomState(GLOBAL_SEED)
    flat = cover.reshape(-1).astype(np.int16)
    seg = flat[:total].copy()
    delta = rng.choice(np.array([-1, 1], dtype=np.int16), size=total)
    delta = np.where(seg == 0, 1, np.where(seg == 255, -1, delta)).astype(np.int16)
    flat[:total] = np.where((seg & 1) != bits, seg + delta, seg)
    return flat.astype(np.uint8).reshape(cover.shape), {"depth": 1}

def lsb_matching_extract(stego, meta):
    total = int(meta["total_payload_bits"])
    return (stego.reshape(-1)[:total] & 1).astype(np.uint8)

BASELINE_EMBED["LSB_Matching"] = lsb_matching_embed
EXTRACTORS["LSB_Matching"] = lsb_matching_extract
_r = process_pair("LSB_Matching", test_pairs[0], 64)
assert _r["status"] == "PASS", _r
_r512 = process_pair("LSB_Matching", test_pairs[0], 512)
print(f"LSB Matching self-test (64x64): {_r['status']} | PSNR {_r['psnr']:.3f} dB | BER {_r['ber']:.6f}")
print(f"LSB Matching at 512x512: {_r512['status']} - {_r512['note']}")

In [ ]:
# STEP 34 — Pixel Value Differencing (Wu-Tsai ranges) on horizontal pixel pairs of each colour channel
PVD_RANGES = [(0, 7), (8, 15), (16, 31), (32, 63), (64, 127), (128, 255)]
PVD_LOWER = np.array([r[0] for r in PVD_RANGES], dtype=np.int64)
PVD_NBITS = np.array([int(math.log2(r[1] - r[0] + 1)) for r in PVD_RANGES], dtype=np.int64)

def pvd_split_pairs(img):
    arr = img.transpose(2, 0, 1).astype(np.int64)                   # channel-major, row-major
    return arr[:, :, 0::2].reshape(-1), arr[:, :, 1::2].reshape(-1)

def pvd_merge_pairs(p1, p2, shape):
    H, W, C = shape
    arr = np.zeros((C, H, W), dtype=np.int64)
    arr[:, :, 0::2] = p1.reshape(C, H, W // 2)
    arr[:, :, 1::2] = p2.reshape(C, H, W // 2)
    return arr.transpose(1, 2, 0).astype(np.uint8)

def pvd_embed(cover, bits):
    total = len(bits)
    p1, p2 = pvd_split_pairs(cover)
    d = np.abs(p1 - p2)
    idx = np.searchsorted(PVD_LOWER, d, side="right") - 1
    nb, lower = PVD_NBITS[idx], PVD_LOWER[idx]
    capacity = int(nb.sum())
    if capacity < total:
        raise CapacityError(f"payload {total} bits exceeds the PVD capacity {capacity} bits of this cover")
    q1, q2 = p1.copy(), p2.copy()
    bits_l, skipped, pos = np.asarray(bits, dtype=np.uint8).tolist(), [], 0
    nb_l, lower_l, d_l, p1_l, p2_l = nb.tolist(), lower.tolist(), d.tolist(), p1.tolist(), p2.tolist()
    for k in range(len(p1_l)):
        if pos >= total:
            break
        n = nb_l[k]
        chunk = bits_l[pos:pos + n]
        b = 0
        for x in chunk:
            b = (b << 1) | x
        b <<= (n - len(chunk))
        m = lower_l[k] + b - d_l[k]
        a, c = p1_l[k], p2_l[k]
        if m > 0:
            h1, h2 = (m + 1) // 2, m // 2
            na, nc = (a + h1, c - h2) if a >= c else (a - h1, c + h2)
        else:
            mm = -m; h1, h2 = (mm + 1) // 2, mm // 2
            na, nc = (a - h1, c + h2) if a >= c else (a + h1, c - h2)
        if not (0 <= na <= 255 and 0 <= nc <= 255):
            skipped.append(k); continue                                   # recorded in the metadata
        q1[k], q2[k] = na, nc
        pos += n
    if pos < total:
        raise CapacityError(f"payload {total} bits exceeds the PVD capacity after skipping {len(skipped)} overflow pairs")
    return pvd_merge_pairs(q1, q2, cover.shape), {"skipped_pairs": [int(x) for x in skipped], "pvd_ranges": PVD_RANGES}

def pvd_extract(stego, meta):
    total = int(meta["total_payload_bits"])
    p1, p2 = pvd_split_pairs(stego)
    d = np.abs(p1 - p2)
    idx = np.searchsorted(PVD_LOWER, d, side="right") - 1
    nb, lower = PVD_NBITS[idx], PVD_LOWER[idx]
    b = d - lower
    j = np.arange(7)[None, :]
    mat = (b[:, None] >> np.maximum(nb[:, None] - 1 - j, 0)) & 1
    mask = j < nb[:, None]
    valid = np.ones(len(p1), dtype=bool)
    valid[np.asarray(meta.get("skipped_pairs", []), dtype=np.int64)] = False
    bits = mat[mask & valid[:, None]].astype(np.uint8)
    if len(bits) < total:
        bits = np.concatenate([bits, np.zeros(total - len(bits), dtype=np.uint8)])
    return bits[:total]

BASELINE_EMBED["PVD"] = pvd_embed
EXTRACTORS["PVD"] = pvd_extract
_r = process_pair("PVD", test_pairs[0], 64)
assert _r["status"] == "PASS", _r
_r2 = process_pair("PVD", test_pairs[0], 256)
_r512 = process_pair("PVD", test_pairs[0], 512)
print(f"PVD self-test (64x64): {_r['status']} | PSNR {_r['psnr']:.3f} dB | BER {_r['ber']:.6f}")
print(f"PVD at 256x256: {_r2['status']} | PVD at 512x512: {_r512['status']} - {_r512['note']}")

In [ ]:
# STEP 34B — EMD (Exploiting Modification Direction), canonical n = 2: each pair of samples carries one base-5 digit.
EMD_N = 2
EMD_BASE = 2 * EMD_N + 1
EMD_WORD_BITS, EMD_WORD_DIGITS = 16, 7          # 16 payload bits -> 7 base-5 digits (5**7 = 78125 >= 2**16)

def bits_to_digits(bits):
    b = np.asarray(bits, dtype=np.int64)
    assert len(b) % EMD_WORD_BITS == 0
    words = (b.reshape(-1, EMD_WORD_BITS) * (1 << np.arange(EMD_WORD_BITS - 1, -1, -1))).sum(axis=1)
    digits = np.zeros((len(words), EMD_WORD_DIGITS), dtype=np.int64)
    for j in range(EMD_WORD_DIGITS - 1, -1, -1):
        digits[:, j] = words % EMD_BASE; words = words // EMD_BASE
    return digits.reshape(-1)

def digits_to_bits(digits):
    d = np.asarray(digits, dtype=np.int64).reshape(-1, EMD_WORD_DIGITS)
    words = np.zeros(len(d), dtype=np.int64)
    for j in range(EMD_WORD_DIGITS):
        words = words * EMD_BASE + d[:, j]
    return ((words[:, None] >> np.arange(EMD_WORD_BITS - 1, -1, -1)) & 1).astype(np.uint8).reshape(-1)

def emd_capacity_bits(cover):
    return int((cover.size // EMD_N) // EMD_WORD_DIGITS * EMD_WORD_BITS)

def emd_embed(cover, bits):
    total = len(bits)
    if total % EMD_WORD_BITS:
        raise CapacityError(f"payload of {total} bits is not a multiple of {EMD_WORD_BITS}")
    digits = bits_to_digits(bits); nd = len(digits)
    if nd * EMD_N > cover.size:
        raise CapacityError(f"payload {total} bits needs {nd} digits ({nd * EMD_N} samples); EMD (n={EMD_N}) capacity of this cover is {emd_capacity_bits(cover)} bits")
    flat = cover.reshape(-1).astype(np.int64).copy()
    g = flat[:nd * EMD_N].reshape(nd, EMD_N)                            # view: edits write through to flat
    f = (g * np.arange(1, EMD_N + 1)).sum(axis=1) % EMD_BASE
    s = (digits - f) % EMD_BASE
    inc, dec = (s >= 1) & (s <= EMD_N), s > EMD_N
    pos = np.where(inc, s - 1, np.where(dec, 2 * EMD_N - s, 0))
    delta = np.where(inc, 1, np.where(dec, -1, 0))
    idx = np.arange(nd)
    newv = g[idx, pos] + delta
    bad = (newv < 0) | (newv > 255)
    g[idx[~bad], pos[~bad]] = newv[~bad]
    for k in np.flatnonzero(bad):                                       # saturated samples: smallest feasible change with the same residue
        a, c, target, best = int(g[k, 0]), int(g[k, 1]), int(s[k]), None
        for d1 in range(-3, 4):
            for d2 in range(-3, 4):
                if (d1 + 2 * d2) % EMD_BASE == target and 0 <= a + d1 <= 255 and 0 <= c + d2 <= 255:
                    cost = d1 * d1 + d2 * d2
                    if best is None or cost < best[0]: best = (cost, d1, d2)
        g[k, 0] += best[1]; g[k, 1] += best[2]
    assert flat.min() >= 0 and flat.max() <= 255
    return flat.astype(np.uint8).reshape(cover.shape), {"emd_n": EMD_N, "num_digits": int(nd)}

def emd_extract(stego, meta):
    n, nd, total = int(meta["emd_n"]), int(meta["num_digits"]), int(meta["total_payload_bits"])
    g = stego.reshape(-1)[:nd * n].astype(np.int64).reshape(nd, n)
    digits = (g * np.arange(1, n + 1)).sum(axis=1) % (2 * n + 1)
    return digits_to_bits(digits)[:total]

# ---- round-trip tests before any evaluation ----
rng_e = np.random.RandomState(GLOBAL_SEED)
_b = rng_e.randint(0, 2, size=4096).astype(np.uint8)
assert np.array_equal(digits_to_bits(bits_to_digits(_b)), _b)
for name, cov in [("random", rng_e.randint(0, 256, (512, 512, 3)).astype(np.uint8)),
                  ("all 255", np.full((512, 512, 3), 255, np.uint8)), ("all 0", np.zeros((512, 512, 3), np.uint8))]:
    bits = rng_e.randint(0, 2, size=32768).astype(np.uint8)
    st, par = emd_embed(cov, bits)
    assert np.array_equal(emd_extract(st, {"total_payload_bits": len(bits), **par}), bits), f"EMD round trip failed ({name})"
    diff = np.abs(st.astype(int) - cov.astype(int))
    print(f"EMD round trip OK ({name}): modified samples {int((diff > 0).sum())} | max change {int(diff.max())}")

BASELINE_EMBED["EMD"] = emd_embed
EXTRACTORS["EMD"] = emd_extract
BASELINE_CAPACITY["EMD"] = emd_capacity_bits
if "EMD" not in ALGORITHMS:
    ALGORITHMS.append("EMD")
ALGO_LABELS["EMD"] = "EMD (n=2)"
if "modified_samples" not in RESULT_COLUMNS:
    RESULT_COLUMNS.append("modified_samples")
if not getattr(fill_metrics, "_wrapped", False):                       # record the number of modified samples for EVERY method
    _orig_fill_metrics = fill_metrics
    def fill_metrics(res, cover, stego, secret_img, secret_bits, rec_bits):
        rec_img = _orig_fill_metrics(res, cover, stego, secret_img, secret_bits, rec_bits)
        res["modified_samples"] = int(np.count_nonzero(cover != stego))
        return rec_img
    fill_metrics._wrapped = True

_r = process_pair("EMD", test_pairs[0], 64);  assert _r["status"] == "PASS", _r
_r2 = process_pair("EMD", test_pairs[0], 256); assert _r2["status"] == "PASS", _r2
_r5 = process_pair("EMD", test_pairs[0], 512)
print(f"EMD 64x64: PSNR {_r['psnr']:.3f} dB, modified samples {_r['modified_samples']} | 256x256: PSNR {_r2['psnr']:.3f} dB | 512x512: {_r5['status']}")
print("EMD capacity of a 512x512 RGB cover:", emd_capacity_bits(np.zeros((512, 512, 3), np.uint8)), "bits")

In [ ]:
# STEP 35A — What were the v2 baseline ERROR rows? (reads v2 files only; nothing is written)
for m in ("LSB_Replacement", "LSB_Matching", "PVD"):
    for s in (64, 128, 256):
        path = os.path.join(OLD_ROOT, "comparison_results", "per_sample", f"{m}_s{s}_base.csv")
        if not os.path.exists(path):
            print(f"{m} {s}x{s}: file not found"); continue
        df = pd.read_csv(path)
        notes = df.note.dropna().astype(str).value_counts().head(2).to_dict()
        print(f"{m} {s}x{s}: status {df.status.value_counts().to_dict()} | file written {time.ctime(os.path.getmtime(path))} | notes {notes}")

In [ ]:
# STEP 35 — Resumable evaluation driver: every algorithm uses exactly the same assigned test pairs.
# Saved ERROR rows are never reused as results: they are re-evaluated. Three consecutive errors stop the run with the first note.
EVAL_PAIR_LIST = test_pairs if EVAL_PAIRS is None else test_pairs[:EVAL_PAIRS]
EVAL_IDS = [p["pair_id"] for p in EVAL_PAIR_LIST]
assert all(p["split"] == "test" for p in EVAL_PAIR_LIST)

def flatten_result(r):
    r = dict(r)
    for k in ("lsb_usage", "dna_rule_usage"):
        if isinstance(r.get(k), dict):
            r[k] = json.dumps({str(a): int(b) for a, b in r[k].items()})
    return r

def _evaluate_one(method, i, size, state):
    r = flatten_result(process_pair(method, EVAL_PAIR_LIST[i], size))
    state["consecutive"] = state["consecutive"] + 1 if r["status"] == "ERROR" else 0
    if state["consecutive"] >= 3:
        raise RuntimeError(f"{method} {size}x{size}: 3 consecutive implementation errors. First note: {r['note']}")
    return r

def evaluate_method(method, size):
    tag = MODEL_SHA256[:8] if method == "Proposed_DNA_PPO" else "base"
    path = os.path.join(PER_SAMPLE_DIR, f"{method}_s{size}_{tag}.csv")
    rows, state = [], {"consecutive": 0}
    if RESUME_EVAL and os.path.exists(path):
        prev = pd.read_csv(path)
        assert prev.pair_id.tolist() == EVAL_IDS[:len(prev)], "Saved results belong to a different pair list."
        rows = prev.to_dict("records")
        bad = [i for i, r in enumerate(rows) if r.get("status") == "ERROR"]
        if bad:
            print(f"[{size}x{size}] {method}: re-evaluating {len(bad)} saved ERROR rows.")
            for i in bad:
                rows[i] = _evaluate_one(method, i, size, state)
        if len(rows) >= len(EVAL_IDS):
            df = pd.DataFrame(rows[:len(EVAL_IDS)]).reindex(columns=RESULT_COLUMNS); df.to_csv(path, index=False)
            print(f"[{size}x{size}] {method}: loaded {len(EVAL_IDS)} saved results. Status counts: {df.status.value_counts().to_dict()}")
            return df
        print(f"[{size}x{size}] {method}: resuming after {len(rows)} saved pairs.")
    else:
        print(f"[{size}x{size}] {method}: evaluating {len(EVAL_IDS)} test pairs...")
    t0 = time.perf_counter()
    for i in range(len(rows), len(EVAL_PAIR_LIST)):
        rows.append(_evaluate_one(method, i, size, state))
        if (i + 1) % 5 == 0 or i + 1 == len(EVAL_PAIR_LIST):
            pd.DataFrame(rows).reindex(columns=RESULT_COLUMNS).to_csv(path, index=False)
            print(f"[{size}x{size}] {method}: {i+1}/{len(EVAL_PAIR_LIST)} pairs | elapsed {time.perf_counter()-t0:.0f} s")
    df = pd.DataFrame(rows).reindex(columns=RESULT_COLUMNS)
    df.to_csv(path, index=False)
    print(f"[{size}x{size}] {method}: completed. Status counts: {df.status.value_counts().to_dict()}")
    return df

print(f"Evaluation pairs: {len(EVAL_PAIR_LIST)} (same pairs for all algorithms and all secret sizes). Algorithms: {[ALGO_LABELS[m] for m in ALGORITHMS]}")

In [ ]:
# STEP 36 — Clean experiments: all four algorithms, all four secret sizes, same pairs
frames = []
for size in SECRET_SIZES:
    for method in ALGORITHMS:
        frames.append(evaluate_method(method, size))
all_results_df = pd.concat(frames, ignore_index=True)
all_results_df.to_csv(os.path.join(COMPARISON_DIR, "all_results_clean.csv"), index=False)
for s in SECRET_SIZES:
    ref_ids = all_results_df[(all_results_df.method == ALGORITHMS[0]) & (all_results_df.secret_size == s)].pair_id.tolist()
    for m in ALGORITHMS:
        assert all_results_df[(all_results_df.method == m) & (all_results_df.secret_size == s)].pair_id.tolist() == ref_ids
print("Clean experiments completed. Result rows:", len(all_results_df))
print(all_results_df.groupby(["secret_size", "method"]).status.value_counts().unstack(fill_value=0).to_string())

In [ ]:
# STEP 36B — Implementation errors must be zero. Capacity failures are reported separately and are legitimate results.
errs = all_results_df[all_results_df.status == "ERROR"]
if len(errs):
    print(errs.groupby(["secret_size", "method"]).note.agg(["count", "first"]).to_string())
    raise RuntimeError("Implementation errors found. Fix them before using any comparison table.")
print(all_results_df.groupby(["secret_size", "method"]).status.value_counts().unstack(fill_value=0).to_string())
print("No implementation errors. FAILED_CAPACITY rows are payloads that the algorithm cannot carry (never padded or shrunk).")

In [ ]:
# STEP 31B — Receiver validation in a fresh process from copies of ONLY the saved stego image, metadata, model and receiver file
RECV_PAIRS_PER_SIZE = 3
RECV_DIR = os.path.join(ROOT, "receiver_validation")
rows = []
for s in SECRET_SIZES:
    ok_rows = all_results_df[(all_results_df.method == "Proposed_DNA_PPO") & (all_results_df.secret_size == s) &
                             (all_results_df.status == "PASS")].head(RECV_PAIRS_PER_SIZE)
    assert len(ok_rows) > 0, f"No passed proposed-method pairs at {s}x{s}."
    for rec in ok_rows.to_dict("records"):
        pair = pairs_by_id[rec["pair_id"]]
        d = os.path.join(RECV_DIR, f"s{s}", rec["pair_id"]); assert_in_root(os.path.join(d, "x")); os.makedirs(d, exist_ok=True)
        for src, name in [(rec["stego_path"], "stego.png"), (rec["metadata_path"], "meta.json"),
                          (DEPLOYED_MODEL_ZIP, "model.zip"), (RECEIVER_FILE, "receiver.py")]:
            shutil.copyfile(src, os.path.join(d, name))
        proc = subprocess.run([sys.executable, "receiver.py", "stego.png", "meta.json", "recovered.png", "model.zip"],
                              cwd=d, capture_output=True, text=True)
        assert proc.returncode == 0, f"{s}x{s} {rec['pair_id']}: receiver failed: {proc.stdout} {proc.stderr[-400:]}"
        child = json.loads([ln for ln in proc.stdout.strip().splitlines() if ln.startswith("{")][-1])
        with open(os.path.join(d, "meta.json")) as f: meta = json.load(f)
        truth = load_secret_image(pair[f"secret_s{s}"], s)
        rec_img = np.array(Image.open(os.path.join(d, "recovered.png")))
        exact = bool(rec_img.shape == truth.shape and np.array_equal(rec_img, truth))
        crc_ok = (zlib.crc32(np.packbits(np.unpackbits(rec_img.reshape(-1))).tobytes()) & 0xFFFFFFFF) == meta["payload_crc32"]
        blocks_ok = child["blocks_used"] == len(meta["records"]) == meta["num_records"]
        payload_ok = sum(r["bits_embedded"] for r in meta["records"]) == meta["total_payload_bits"] == s * s * 8
        model_ok = bool(child["model_check"].get("checked")) and child["model_check"]["sha256"] == PPO_MODEL_INFO["sha256"]
        rejected = 0
        m0 = meta
        def _drop(m): m.update(records=m["records"][:-1])
        def _rule(m): m["records"][0].update(dna_rule=(m["records"][0]["dna_rule"] + 1) % 8)
        def _model(m): m["ppo_model"]["sha256"] = "0" * 64
        def _missing(m): m.pop("total_payload_bits")
        def _badtype(m): m["records"][0]["lsb_depth"] = "3"
        for tamper in (_drop, _rule, _model, _missing, _badtype):
            mm = json.loads(json.dumps(m0)); tamper(mm)
            try:
                receiver_extract(os.path.join(d, "stego.png"), mm, model_path=DEPLOYED_MODEL_ZIP)
            except ReceiverError:
                rejected += 1
        rows.append({"size": s, "pair": rec["pair_id"], "exact image": exact, "CRC32 ok": bool(child["crc_ok"] is True and crc_ok),
                     "model verified": model_ok, "records / blocks consistent": bool(blocks_ok), "payload length ok": payload_ok,
                     "tampered metadata rejected (of 5)": rejected})
rv = pd.DataFrame(rows)
rv.to_csv(os.path.join(TABLE_DIR, "receiver_validation.csv"), index=False)
print(rv.to_string(index=False))
assert rv[["exact image", "CRC32 ok", "model verified", "records / blocks consistent", "payload length ok"]].all().all()
assert (rv["tampered metadata rejected (of 5)"] == 5).all()
print("Independent receiver validation passed for all four secret sizes.")

In [ ]:
# STEP 37 — Noise generation (applied to a copy of the saved stego image, never to the original)
def add_noise(img, kind, level, rng):
    x = img.astype(np.float64)
    if kind == "Gaussian":
        y = x + rng.normal(0.0, level, x.shape)
    elif kind == "Salt-and-pepper":
        y = x.copy(); r = rng.random_sample(x.shape)
        y[r < level / 2] = 0; y[(r >= level / 2) & (r < level)] = 255
    elif kind == "Speckle":
        y = x + x * rng.normal(0.0, math.sqrt(level), x.shape)
    else:
        raise ValueError(kind)
    return np.clip(np.rint(y), 0, 255).astype(np.uint8)

_t = add_noise(cover_image, "Gaussian", 1.0, np.random.RandomState(1))
assert _t.shape == cover_image.shape and _t.dtype == np.uint8 and not np.array_equal(_t, cover_image)
print("Noise types:", list(NOISE_LEVELS), "| levels:", NOISE_LEVELS)

In [ ]:
# STEP 37A — Noise diagnostics. Measures (does not assume) why bits are lost. Uses test pairs only for evaluation-side analysis.
import io
from scipy.special import ndtr

# (1) Is the noise applied correctly? measured altered-sample fraction vs the analytic prediction
base_img = load_cover_image(test_pairs[0]["cover_path"])
rows = []
for kind, levels in NOISE_LEVELS.items():
    for li, lvl in enumerate(levels):
        noisy = add_noise(base_img, kind, lvl, np.random.RandomState(NOISE_SEED + li))
        assert noisy.dtype == np.uint8 and noisy.shape == base_img.shape
        buf = io.BytesIO(); Image.fromarray(noisy).save(buf, format="PNG"); buf.seek(0)
        assert np.array_equal(np.array(Image.open(buf).convert("RGB")), noisy), "PNG saving/RGB conversion changed the noisy image."
        if kind == "Gaussian":
            pred = 2 * (1 - ndtr(0.5 / lvl))
        elif kind == "Salt-and-pepper":
            pred = lvl
        else:
            sd = base_img.astype(np.float64) * math.sqrt(lvl)
            pred = float(np.mean(np.where(sd > 0, 2 * (1 - ndtr(0.5 / np.maximum(sd, 1e-9))), 0.0)))
        rows.append({"noise": kind, "level": lvl, "predicted altered fraction": pred, "measured altered fraction": float(np.mean(noisy != base_img))})
print("(1) Noise generation check (PNG save / RGB conversion verified lossless):")
print(pd.DataFrame(rows).round(5).to_string(index=False))

# (2) Same noise for every algorithm? Gaussian noise added to two different stego images must be identical on unclipped pixels
A = load_cover_image(test_pairs[0]["cover_path"]); B = load_cover_image(test_pairs[1]["cover_path"])
nA = add_noise(A, "Gaussian", 2.0, np.random.RandomState(7)).astype(np.int16) - A
nB = add_noise(B, "Gaussian", 2.0, np.random.RandomState(7)).astype(np.int16) - B
inner = (A >= 20) & (A <= 235) & (B >= 20) & (B <= 235)
assert np.array_equal(nA[inner], nB[inner]), "The same seed did not give the same noise field."
print("(2) Fairness: identical seeds give identical noise fields for different images (salt-and-pepper masks and speckle draws use the same random stream).")

# (3) DNA mapping: does the reverse transform multiply single raw-bit errors?
amp = []
for d in range(NUM_DNA_RULES):
    lut = dna_transform_lut[d, BASE_DNA_RULE]
    dist = [bin(int(lut[w]) ^ int(lut[w ^ e])).count("1") for w in range(4) for e in (1, 2)]
    amp.append({"DNA rule": d, "mean payload bit errors per raw bit error": float(np.mean(dist)), "max": max(dist)})
print("(3) DNA reverse mapping: payload bit errors caused by ONE raw bit error")
print(pd.DataFrame(amp).round(3).to_string(index=False))

# (4) Where do the errors occur? Proposed method, Gaussian sigma = 0.5, 64x64 secrets, first 3 test pairs
by_depth_pos, by_sig = {}, np.zeros(8); n_sig = 0; raw_err_total = 0; pay_err_total = 0
for pair in test_pairs[:3]:
    res, stego, meta = run_sender_proposed(pair, 64, make_ppo_chooser(ppo_model), PPO_MODEL_INFO)
    assert res["status"] == "SENT"
    smp = prepare_sample(pair, 64)
    noisy = add_noise(stego, "Gaussian", 0.5, np.random.RandomState(NOISE_SEED))
    cb, nb = image_to_blocks(stego), image_to_blocks(noisy)
    for r in meta["records"]:
        # signature: extract_bits_from_block(block, lsb_amount, location_id, pixel_locations_, n_bits_needed)
        rc = extract_bits_from_block(cb[r["block_index"]], r["lsb_depth"], r["pixel_location"], pixel_locations, r["bits_embedded"])
        rn = extract_bits_from_block(nb[r["block_index"]], r["lsb_depth"], r["pixel_location"], pixel_locations, r["bits_embedded"])
        err = (rc != rn)
        pos = np.arange(len(err)) % r["lsb_depth"]
        for ps in range(r["lsb_depth"]):
            e = by_depth_pos.setdefault((r["lsb_depth"], ps), [0, 0])
            e[0] += int(err[pos == ps].sum()); e[1] += int((pos == ps).sum())
        raw_err_total += int(err.sum())
    pay = receiver_extract(noisy, meta, model_path=None, strict_crc=False)["payload_bits"]
    perr = (pay != smp["secret_bits"]); pay_err_total += int(perr.sum())
    by_sig += perr.reshape(-1, 8).sum(axis=0); n_sig += perr.reshape(-1, 8).shape[0]
print("(4a) Raw bit error rate by LSB depth and bit position inside the embedded group (0 = most significant embedded bit):")
print(pd.DataFrame([{"LSB depth": k[0], "position in group": k[1], "bit error rate": v[0] / max(1, v[1]), "bits": v[1]}
                    for k, v in sorted(by_depth_pos.items())]).round(4).to_string(index=False))
print(f"(4b) Payload errors / raw errors (DNA + packing amplification): {pay_err_total / max(1, raw_err_total):.3f}")
print("(4c) Payload error rate by secret-pixel bit significance (bit 7 = most significant):")
print(pd.DataFrame({"secret bit": [7, 6, 5, 4, 3, 2, 1, 0], "error rate": by_sig / n_sig}).round(4).to_string(index=False))

# (5) PVD: do bit errors grow along the stream (synchronisation loss) compared with LSB replacement?
dec = {"PVD": [], "LSB Replacement": []}; range_changed = []
for pair in test_pairs[:3]:
    cover, _, bits = load_pair_arrays(pair, 64)
    st_p, par_p = pvd_embed(cover, bits); st_l, par_l = lsb_replacement_embed(cover, bits)
    nz_p = add_noise(st_p, "Gaussian", 0.25, np.random.RandomState(NOISE_SEED))
    nz_l = add_noise(st_l, "Gaussian", 0.25, np.random.RandomState(NOISE_SEED))
    e_p = pvd_extract(nz_p, {"total_payload_bits": len(bits), **par_p}) != bits
    e_l = lsb_replacement_extract(nz_l, {"depth": par_l["depth"], "total_payload_bits": len(bits)}) != bits
    n = len(bits); tenth = max(1, n // 10)
    dec["PVD"].append([e_p[i * tenth:(i + 1) * tenth].mean() for i in range(10)])
    dec["LSB Replacement"].append([e_l[i * tenth:(i + 1) * tenth].mean() for i in range(10)])
    a1, a2 = pvd_split_pairs(st_p); b1, b2 = pvd_split_pairs(nz_p)
    ia = np.searchsorted(PVD_LOWER, np.abs(a1 - a2), side="right") - 1
    ib = np.searchsorted(PVD_LOWER, np.abs(b1 - b2), side="right") - 1
    range_changed.append(float(np.mean(ia[:n // 4] != ib[:n // 4])))
print("(5) BER per tenth of the payload stream (Gaussian sigma 0.25, 64x64):")
print(pd.DataFrame({k: np.mean(v, axis=0) for k, v in dec.items()}).round(4).T.to_string())
print(f"    PVD pairs whose difference-range index changed: {100*np.mean(range_changed):.1f}% -> bit counts shift and later bits are misread.")

In [ ]:
# STEP 37B — Controlled experiment (A and C). Same pairs, same noise; only the embedding policy differs. No training on test data.
NOISE_EXP_PAIRS, NOISE_EXP_SIZES = 8, [64, 128]
NOISE_EXP_SETTINGS = [("Gaussian", 0.25), ("Gaussian", 0.5), ("Salt-and-pepper", 0.001), ("Salt-and-pepper", 0.01),
                      ("Speckle", 1e-5), ("Speckle", 1e-4)]
POLICY_EXP_FILE = os.path.join(NOISE_RESULT_DIR, "policy_noise_experiment.csv")
policy_factories = {
    "PPO (deployed)":             lambda smp, i: make_ppo_chooser(ppo_model),
    "Random valid":               lambda smp, i: make_random_valid_chooser(GLOBAL_SEED + i),
    "Fixed 3-LSB textured-first": lambda smp, i: make_block_order_chooser(smp, 3, "textured"),
    "Fixed 1-LSB textured-first": lambda smp, i: make_block_order_chooser(smp, 1, "textured"),
    "Fixed 3-LSB darkest-first":  lambda smp, i: make_block_order_chooser(smp, 3, "dark"),
    "Fixed 1-LSB darkest-first":  lambda smp, i: make_block_order_chooser(smp, 1, "dark"),
}
if RESUME_EVAL and os.path.exists(POLICY_EXP_FILE):
    policy_df = pd.read_csv(POLICY_EXP_FILE)
    print("Policy-noise experiment loaded from", POLICY_EXP_FILE)
else:
    rows = []
    for s in NOISE_EXP_SIZES:
        for i, pair in enumerate(test_pairs[:NOISE_EXP_PAIRS]):
            smp = prepare_sample(pair, s)
            for pname, fac in policy_factories.items():
                env, _ = run_embedding_episode(smp, fac(smp, i))
                recs = records_from_log(env.block_bit_log)
                ok, msg = check_records(recs, len(smp["payload_bits"]))
                assert ok, (pname, msg)
                stego = blocks_to_image(env.stego_blocks, smp["cover_image"].shape, BLOCK_SIZE)
                meta = build_metadata_proposed(smp, recs, None)
                for ki, (kind, lvl) in enumerate(NOISE_EXP_SETTINGS):
                    noisy = add_noise(stego, kind, lvl, np.random.RandomState(NOISE_SEED + 1009 * i + 31 * ki))
                    rb = receiver_extract(noisy, meta, model_path=None, strict_crc=False)["payload_bits"]
                    bm = bit_metrics(smp["secret_bits"], rb)
                    q = secret_quality(smp["secret_image"], np.packbits(rb).reshape(s, s).astype(np.uint8))
                    rows.append({"size": s, "pair": pair["pair_id"], "policy": pname, "noise": kind, "level": lvl,
                                 "blocks": len(recs), "cover_psnr": calculate_psnr(smp["cover_image"], stego),
                                 "ber": bm["ber"], "bit_accuracy": bm["bit_accuracy"],
                                 "secret_psnr": q["secret_psnr"], "secret_ssim": q["secret_ssim"]})
            print(f"[{s}x{s}] pair {i+1}/{NOISE_EXP_PAIRS} done")
    policy_df = pd.DataFrame(rows); policy_df.to_csv(POLICY_EXP_FILE, index=False)

pv = policy_df.replace([np.inf, -np.inf], np.nan)
tab = pv.groupby(["size", "policy", "noise", "level"]).agg(ber=("ber", "mean"), secret_ssim=("secret_ssim", "mean"),
                                                         cover_psnr=("cover_psnr", "mean"), blocks=("blocks", "mean")).round(5)
print(tab.to_string())
print("\nMean BER by policy and noise setting (lower is better); compare each policy with 'PPO (deployed)':")
print(pv.pivot_table(index=["size", "policy"], columns=["noise", "level"], values="ber", aggfunc="mean").round(4).to_string())
print("Cover PSNR by policy (the image-quality cost of each policy):")
print(pv.groupby(["size", "policy"]).cover_psnr.mean().round(2).to_string())

In [ ]:
# STEP 37C — Consistency guard: the in-memory clean results must belong to THIS experiment's test pairs
_ids_in_results = set(all_results_df.pair_id)
_stale = _ids_in_results - set(pairs_by_id)
assert not _stale, (f"all_results_df contains pair IDs from another experiment (e.g. {sorted(_stale)[:3]}). "
                    f"Re-run STEP 35 and STEP 36 in order, then STEP 38.")
assert _ids_in_results == set(EVAL_IDS), "all_results_df does not cover exactly the current EVAL_PAIR_LIST. Re-run STEP 35 and 36."
assert all(os.path.abspath(p).startswith(_new_root + os.sep) for p in all_results_df.stego_path.dropna()), \
    "Result paths point outside the new output directory. Re-run STEP 35 and 36."
print(f"Guard passed: {len(EVAL_IDS)} evaluation pairs, results belong to {os.path.basename(_new_root)}.")

In [ ]:
# STEP 38 — Noise experiments: noisy copy of the saved stego image + original metadata -> extraction -> compare with the original secret.
# Extraction completed, BER, bit accuracy, exact recovery and recovered-image quality are reported separately.
def noise_trial(method, noisy, stego_clean, meta, secret_img, secret_bits, size):
    row = {"noisy_vs_stego_psnr": calculate_psnr(stego_clean, noisy)}
    t0 = time.perf_counter()
    try:
        rbits = extract_with_method(method, noisy, meta)
        el = time.perf_counter() - t0
        m = bit_metrics(secret_bits, rbits)
        rec_img = np.packbits(rbits).reshape(size, size).astype(np.uint8) if len(rbits) == size * size * 8 else None
        q = secret_quality(secret_img, rec_img) if rec_img is not None else \
            {"secret_mse": float("nan"), "secret_psnr": float("nan"), "secret_ssim": float("nan"), "secret_pixel_accuracy": float("nan")}
        row.update(extraction_completed=True, ber=m["ber"], bit_accuracy=m["bit_accuracy"], exact_recovery=m["exact_recovery"],
                   extraction_time_s=el, error="", **q)
    except Exception as e:
        rec_img = None
        row.update(extraction_completed=False, ber=float("nan"), bit_accuracy=float("nan"), exact_recovery=False,
                   extraction_time_s=float("nan"), error=repr(e), secret_mse=float("nan"), secret_psnr=float("nan"),
                   secret_ssim=float("nan"), secret_pixel_accuracy=float("nan"))
    return row, rec_img

def run_noise_experiment(method, size, si):
    path = os.path.join(NOISE_RESULT_DIR, f"noise_{method}_s{size}_{MODEL_SHA256[:8] if method == 'Proposed_DNA_PPO' else 'base'}.csv")
    sub = all_results_df[(all_results_df.method == method) & (all_results_df.secret_size == size) & all_results_df.stego_path.notna()]
    if sub.empty:
        print(f"[{size}x{size}] {method}: no clean stego images exist (see clean status), so the noise experiment is not applicable.")
        return pd.DataFrame([{"method": method, "secret_size": size, "pair_id": "", "noise_type": "N/A", "noise_level": float("nan"),
                              "extraction_completed": False, "error": "no stego image (embedding failed in the clean experiment)"}])
    rows = pd.read_csv(path).to_dict("records") if (RESUME_EVAL and os.path.exists(path)) else []
    done = {r["pair_id"] for r in rows}
    if done:
        print(f"[{size}x{size}] {method} noise: resuming, {len(done)} pairs already completed.")
    t0 = time.perf_counter()
    for k, rec in enumerate(sub.to_dict("records")):
        if rec["pair_id"] in done:
            continue
        pair = pairs_by_id[rec["pair_id"]]
        stego_clean = np.array(Image.open(rec["stego_path"]).convert("RGB"), dtype=np.uint8)
        with open(rec["metadata_path"], "r", encoding="utf-8") as f:
            meta = json.load(f)                                           # original metadata, never modified
        secret_img = load_secret_image(pair[f"secret_s{size}"], size)
        secret_bits = np.unpackbits(secret_img.reshape(-1))
        pidx = EVAL_IDS.index(rec["pair_id"])
        for ti, (kind, levels) in enumerate(NOISE_LEVELS.items()):
            for li, lvl in enumerate(levels):
                rng = np.random.RandomState(NOISE_SEED + 100003 * si + 1009 * pidx + 31 * ti + li)   # same noise for every algorithm
                noisy = add_noise(stego_clean, kind, lvl, rng)
                row, rec_img = noise_trial(method, noisy, stego_clean, meta, secret_img, secret_bits, size)
                row.update(method=method, secret_size=size, pair_id=rec["pair_id"], noise_type=kind, noise_level=lvl)
                if NOISE_SAVE_PAIRS is None or k < NOISE_SAVE_PAIRS:
                    tag = f"{rec['pair_id']}_{kind}_{lvl:g}"
                    d1, d2 = os.path.join(NOISY_STEGO_DIR, method, f"s{size}"), os.path.join(NOISY_RECOVERED_DIR, method, f"s{size}")
                    os.makedirs(d1, exist_ok=True); os.makedirs(d2, exist_ok=True)
                    Image.fromarray(noisy).save(os.path.join(d1, tag + "_noisy_stego.png"))
                    if rec_img is not None:
                        Image.fromarray(rec_img).save(os.path.join(d2, tag + "_recovered.png"))
                rows.append(row)
        if (k + 1) % 5 == 0 or k + 1 == len(sub):
            pd.DataFrame(rows).to_csv(path, index=False)
            print(f"[{size}x{size}] {method} noise: {k+1}/{len(sub)} stego images | elapsed {time.perf_counter()-t0:.0f} s")
    df = pd.DataFrame(rows); df.to_csv(path, index=False)
    return df

noise_frames = []
for si, s in enumerate(SECRET_SIZES):
    for m in ALGORITHMS:
        noise_frames.append(run_noise_experiment(m, s, si))
noise_df = pd.concat(noise_frames, ignore_index=True)
noise_df.to_csv(os.path.join(NOISE_RESULT_DIR, "noise_per_trial_all.csv"), index=False)
print("Noise experiments completed. Trials:", int((noise_df.noise_type != "N/A").sum()))

In [ ]:
# STEP 39 — Result tables (all values come from the experiment outputs)
R = all_results_df.copy()
for c in ["mse", "psnr", "ssim", "ber", "bit_accuracy", "embedding_time_s", "extraction_time_s", "bits_embedded", "payload_bits",
          "payload_rate_bpp", "payload_capacity_bits", "blocks_used", "secret_mse", "secret_psnr", "secret_ssim", "secret_pixel_accuracy"]:
    R[c] = pd.to_numeric(R[c], errors="coerce")
istrue = lambda s: s.map(lambda v: v is True or str(v) == "True")

def mean_ok(series):
    v = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float); v = v[np.isfinite(v)]
    return float(v.mean()) if v.size else float("nan")

def sum_usage(series):
    tot = {1: 0, 2: 0, 3: 0}
    for v in series.dropna():
        if isinstance(v, str):
            for k, c in json.loads(v).items(): tot[int(k)] += int(c)
    return tot

szlab = lambda s: f"{s} x {s}"
rows = []
for s in SECRET_SIZES:
    for m in ALGORITHMS:
        sub = R[(R.method == m) & (R.secret_size == s)]
        emb = sub[sub.stego_path.notna()]
        n = len(sub)
        fail = "" if len(emb) == n else "; ".join(f"{k}: {v}" for k, v in sub[sub.stego_path.isna()].status.value_counts().items())
        u = sum_usage(emb.lsb_usage); ut = sum(u.values())
        rows.append({"Algorithm": ALGO_LABELS[m], "Secret image size": szlab(s), "Cover image size": szlab(COVER_SIZE),
                     "Payload (bits)": expected_payload_bits(s), "Payload rate (bpp)": expected_payload_bits(s) / (COVER_SIZE ** 2),
                     "Capacity (bits, mean)": mean_ok(sub.payload_capacity_bits),
                     "Pairs tested": n, "Pairs embedded": len(emb), "Actual bits embedded (mean)": mean_ok(emb.bits_embedded),
                     "Blocks used (mean)": mean_ok(emb.blocks_used),
                     "LSB-1 blocks (%)": 100 * u[1] / ut if ut else float("nan"), "LSB-2 blocks (%)": 100 * u[2] / ut if ut else float("nan"),
                     "LSB-3 blocks (%)": 100 * u[3] / ut if ut else float("nan"),
                     "MSE": mean_ok(emb.mse), "PSNR (dB)": mean_ok(emb.psnr), "SSIM": mean_ok(emb.ssim),
                     "Embedding time (s)": mean_ok(emb.embedding_time_s),
                     "Extraction completed (%)": round(100 * istrue(sub.extraction_completed).sum() / max(1, n), 2),
                     "BER": mean_ok(emb.ber), "Bit accuracy": mean_ok(emb.bit_accuracy),
                     "Exact recovery rate (%)": round(100 * istrue(sub.exact_recovery).sum() / max(1, n), 2),
                     "Extraction time (s)": mean_ok(emb.extraction_time_s), "Failures": fail})
summary_df = pd.DataFrame(rows)
summary_df.to_csv(os.path.join(TABLE_DIR, "summary_all_sizes.csv"), index=False, encoding="utf-8-sig")
for s in SECRET_SIZES:
    t = summary_df[summary_df["Secret image size"] == szlab(s)]
    t.to_csv(os.path.join(TABLE_DIR, f"comparison_{s}x{s}.csv"), index=False, encoding="utf-8-sig")
    print(f"\n--- Secret image size {szlab(s)} (cover {szlab(COVER_SIZE)}) ---")
    print(t.drop(columns=["Secret image size", "Cover image size"]).round(5).to_string(index=False))

# Noise tables: separate metrics, one table per noise type
N = noise_df[noise_df.noise_type != "N/A"].copy()
for c in ["ber", "bit_accuracy", "extraction_time_s", "secret_mse", "secret_psnr", "secret_ssim", "secret_pixel_accuracy", "noisy_vs_stego_psnr"]:
    N[c] = pd.to_numeric(N[c], errors="coerce")
noise_rows = []
for (m, s, kind, lvl), g in N.groupby(["method", "secret_size", "noise_type", "noise_level"], sort=False):
    noise_rows.append({"Algorithm": ALGO_LABELS[m], "Secret image size": szlab(int(s)), "Noise type": kind, "Noise level": lvl,
                       "Trials": len(g), "Noisy-vs-stego PSNR (dB)": mean_ok(g.noisy_vs_stego_psnr),
                       "Extraction completed (%)": round(100 * istrue(g.extraction_completed).mean(), 2),
                       "BER": mean_ok(g.ber), "BER std": float(np.nanstd(g.ber.to_numpy(dtype=float))),
                       "Bit accuracy": mean_ok(g.bit_accuracy),
                       "Exact recovery (%)": round(100 * istrue(g.exact_recovery).mean(), 2),
                       "Secret pixel accuracy": mean_ok(g.secret_pixel_accuracy), "Secret MSE": mean_ok(g.secret_mse),
                       "Secret PSNR (dB, finite)": mean_ok(g.secret_psnr), "Secret SSIM": mean_ok(g.secret_ssim),
                       "Extraction time (s)": mean_ok(g.extraction_time_s)})
noise_summary_df = pd.DataFrame(noise_rows)
if len(noise_summary_df):
    noise_summary_df.to_csv(os.path.join(TABLE_DIR, "noise_all.csv"), index=False, encoding="utf-8-sig")
    for kind in NOISE_LEVELS:
        t = noise_summary_df[noise_summary_df["Noise type"] == kind]
        t.to_csv(os.path.join(TABLE_DIR, f"noise_{kind.lower().replace('-', '_')}.csv"), index=False, encoding="utf-8-sig")
        print(f"\n--- {kind} noise ---")
        print(t.drop(columns=["Noise type"]).round(5).to_string(index=False))
else:
    print("\nNo noise trials were available, so no noise tables were produced.")
na = noise_df[noise_df.noise_type == "N/A"]
for _, r in na.iterrows():
    print(f"Note: {ALGO_LABELS[r['method']]} at {int(r['secret_size'])}x{int(r['secret_size'])} has no stego image, so no noise results.")
print("\nTables saved in", TABLE_DIR)

In [ ]:
# STEP 39B — Expected number of bit errors per image and the chance of a completely error-free recovery, from the measured BER
T = noise_summary_df.copy()
T["Payload (bits)"] = T["Secret image size"].map(lambda s: expected_payload_bits(int(s.split(" x ")[0])))
T["Expected bit errors per image"] = T["BER"] * T["Payload (bits)"]
T["P(zero bit errors) if errors are independent"] = np.exp(-T["Expected bit errors per image"])
T["Measured exact recovery (%)"] = T["Exact recovery (%)"]
cols = ["Algorithm", "Secret image size", "Noise type", "Noise level", "BER", "Expected bit errors per image",
        "P(zero bit errors) if errors are independent", "Measured exact recovery (%)"]
T[cols].to_csv(os.path.join(TABLE_DIR, "noise_exact_recovery_explained.csv"), index=False, encoding="utf-8-sig")
print(T[T.Algorithm == ALGO_LABELS["Proposed_DNA_PPO"]][cols].round(6).to_string(index=False))
print("\nExact recovery requires zero errors among all payload bits. With a few hundred expected errors per image, "
      "exact recovery is practically impossible even when BER is below 1%; BER, bit accuracy and recovered-image quality "
      "are the meaningful noise metrics.")

In [ ]:
# STEP 39C — Paired comparison on the SAME pairs: proposed method versus every baseline (negative delta = proposed is worse)
from scipy.stats import wilcoxon
R2 = all_results_df.copy()
for c in ["mse", "psnr", "ssim", "modified_samples", "payload_capacity_bits"]:
    R2[c] = pd.to_numeric(R2[c], errors="coerce")
rows = []
for s in SECRET_SIZES:
    P = R2[(R2.method == "Proposed_DNA_PPO") & (R2.secret_size == s) & (R2.status == "PASS")].set_index("pair_id")
    for m in ALGORITHMS[1:]:
        Bm = R2[(R2.method == m) & (R2.secret_size == s) & (R2.status == "PASS")].set_index("pair_id")
        idx = P.index.intersection(Bm.index)
        if len(idx) == 0:
            fails = R2[(R2.method == m) & (R2.secret_size == s)].status.value_counts().to_dict()
            rows.append({"Secret size": szlab(s), "Baseline": ALGO_LABELS[m], "Paired pairs": 0, "Note": f"no embedded pairs: {fails}"}); continue
        dp, ds = P.loc[idx, "psnr"] - Bm.loc[idx, "psnr"], P.loc[idx, "ssim"] - Bm.loc[idx, "ssim"]
        try: pv = float(wilcoxon(dp).pvalue)
        except Exception: pv = float("nan")
        rows.append({"Secret size": szlab(s), "Baseline": ALGO_LABELS[m], "Paired pairs": len(idx),
                     "Proposed PSNR": P.loc[idx, "psnr"].mean(), "Baseline PSNR": Bm.loc[idx, "psnr"].mean(),
                     "dPSNR mean": dp.mean(), "Proposed higher PSNR (%)": 100 * float((dp > 0).mean()), "Wilcoxon p (PSNR)": pv,
                     "Proposed SSIM": P.loc[idx, "ssim"].mean(), "Baseline SSIM": Bm.loc[idx, "ssim"].mean(), "dSSIM mean": ds.mean(),
                     "Proposed MSE": P.loc[idx, "mse"].mean(), "Baseline MSE": Bm.loc[idx, "mse"].mean(),
                     "Proposed modified samples": P.loc[idx, "modified_samples"].mean(),
                     "Baseline modified samples": Bm.loc[idx, "modified_samples"].mean(),
                     "Baseline capacity (bits)": Bm.loc[idx, "payload_capacity_bits"].mean()})
pc = pd.DataFrame(rows)
pc.to_csv(os.path.join(TABLE_DIR, "paired_comparison_vs_baselines.csv"), index=False, encoding="utf-8-sig")
print(pc.round(5).to_string(index=False))

In [ ]:
# STEP 40 — Graphs (only: training progress, PSNR, embedding time, BER versus noise level)
plt.close("all")
hist = pd.read_csv(VAL_HISTORY_FILE)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(hist.timesteps, hist.val_score, marker="o", label="Validation episode return")
ax.axhline(ref_sum["val_score"], color="gray", ls="--", label="Random valid policy")
ax.set_xlabel("Training timesteps"); ax.set_ylabel("Validation score"); ax.grid(alpha=0.3)
ax2 = ax.twinx(); ax2.plot(hist.timesteps, hist.psnr, color="tab:green", marker="s", label="Validation PSNR"); ax2.set_ylabel("PSNR (dB)")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, fontsize=8)
ax.set_title("PPO staged training: validation progress"); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "ppo_training_progress.png"), dpi=150); plt.show()

sizes_lab = [szlab(s) for s in SECRET_SIZES]
def grouped_bar(col, ylabel, title, fname, log_y=False):
    fig, ax = plt.subplots(figsize=(11, 5)); w = 0.8 / len(ALGORITHMS)
    for j, m in enumerate(ALGORITHMS):
        vals = []
        for sz in sizes_lab:
            r = summary_df[(summary_df.Algorithm == ALGO_LABELS[m]) & (summary_df["Secret image size"] == sz)]
            vals.append(float(r[col].iloc[0]) if len(r) else np.nan)
        ax.bar(np.arange(len(sizes_lab)) + (j - (len(ALGORITHMS) - 1) / 2) * w, vals, w, label=ALGO_LABELS[m])
    ax.set_xticks(np.arange(len(sizes_lab))); ax.set_xticklabels(sizes_lab)
    ax.set_xlabel("Secret image size"); ax.set_ylabel(ylabel); ax.set_title(title)
    if log_y: ax.set_yscale("log")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
    fig.text(0.01, 0.01, "Missing bar: the algorithm could not embed this payload (capacity failure).", fontsize=7)
    plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, fname), dpi=150); plt.show()

grouped_bar("PSNR (dB)", "PSNR (dB, cover vs stego)", "PSNR across secret image sizes", "comparison_psnr.png")
grouped_bar("Embedding time (s)", "Embedding time (s)", "Embedding time across secret image sizes", "comparison_embedding_time.png", log_y=True)

if len(noise_summary_df):
    for kind in NOISE_LEVELS:
        fig, axes = plt.subplots(1, len(SECRET_SIZES), figsize=(5 * len(SECRET_SIZES), 4), sharey=True)
        for ax, s in zip(axes, SECRET_SIZES):
            for m in ALGORITHMS:
                d = noise_summary_df[(noise_summary_df["Algorithm"] == ALGO_LABELS[m]) & (noise_summary_df["Secret image size"] == szlab(s))
                                     & (noise_summary_df["Noise type"] == kind)].sort_values("Noise level")
                if len(d): ax.plot(d["Noise level"], d["BER"], marker="o", label=ALGO_LABELS[m])
            ax.set_xscale("log"); ax.set_title(f"{szlab(s)}"); ax.set_xlabel("Noise level"); ax.grid(alpha=0.3)
        axes[0].set_ylabel("BER"); axes[0].legend(fontsize=8)
        plt.suptitle(f"{kind} noise: BER of the recovered payload"); plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f"noise_ber_{kind.lower().replace('-', '_')}.png"), dpi=150); plt.show()
print("Graphs saved in", PLOT_DIR)

In [ ]:
# STEP 41 — Sample outputs: original cover, original secret, stego and recovered secret of the SAME assigned pair
for s in SECRET_SIZES:
    cand = R[(R.method == "Proposed_DNA_PPO") & (R.secret_size == s) & (R.status == "PASS")].head(SAMPLES_PER_SIZE)
    for k, (_, r) in enumerate(cand.iterrows()):
        pair = pairs_by_id[r.pair_id]
        d = os.path.join(SAMPLE_DIR, f"s{s}", r.pair_id); assert_in_root(d + os.sep + "x"); os.makedirs(d, exist_ok=True)
        shutil.copy(pair["cover_path"], os.path.join(d, "original_cover.png"))
        shutil.copy(pair[f"secret_s{s}"], os.path.join(d, "original_secret.png"))
        for m in ALGORITHMS:
            q = R[(R.method == m) & (R.secret_size == s) & (R.pair_id == r.pair_id)]
            if len(q) and isinstance(q.iloc[0].stego_path, str) and isinstance(q.iloc[0].recovered_path, str):
                shutil.copy(q.iloc[0].stego_path, os.path.join(d, f"{m}_stego.png"))
                shutil.copy(q.iloc[0].recovered_path, os.path.join(d, f"{m}_recovered.png"))
        if k == 0:
            cov = np.array(Image.open(os.path.join(d, "original_cover.png")))
            sec = np.array(Image.open(os.path.join(d, "original_secret.png")))
            stg = np.array(Image.open(os.path.join(d, "Proposed_DNA_PPO_stego.png")))
            rec = np.array(Image.open(os.path.join(d, "Proposed_DNA_PPO_recovered.png")))
            fig, axes = plt.subplots(1, 4, figsize=(14, 4))
            for a, im, t, cm in zip(axes, [cov, sec, stg, rec], ["Cover", "Secret", "Stego (proposed)", "Recovered"], [None, "gray", None, "gray"]):
                a.imshow(im, cmap=cm, vmin=0 if cm else None, vmax=255 if cm else None); a.set_title(t); a.axis("off")
            plt.suptitle(f"{szlab(s)} secret | pair {r.pair_id} | PSNR {r.psnr:.2f} dB | BER {r.ber:.1f}")
            plt.tight_layout(); plt.savefig(os.path.join(SAMPLE_DIR, f"overview_s{s}.png"), dpi=120); plt.show()
    print(f"{szlab(s)}: {len(cand)} sample sets saved.")
print("Sample outputs saved in", SAMPLE_DIR)

In [ ]:
# STEP 42 — Final validation and summary
print("Final verification...")
for s in SECRET_SIZES:
    for m in ALGORITHMS:
        sub = R[(R.method == m) & (R.secret_size == s)]
        assert len(sub) == len(EVAL_PAIR_LIST), "The evaluated population differs from the test pair list."
        ok = sub[sub.status == "PASS"]
        if len(ok):
            assert (ok.ber == 0.0).all() and (ok.bit_accuracy == 1.0).all(), f"{m} {s}: a PASS pair has bit errors."
            assert (ok.bits_embedded == expected_payload_bits(s)).all()
            for col in ("stego_path", "metadata_path", "recovered_path"):
                assert ok[col].map(os.path.exists).all() and all(os.path.abspath(p).startswith(_new_root + os.sep) for p in ok[col])
            for p in ok.recovered_path.head(3):
                assert np.array(Image.open(p)).shape == (s, s), "Recovered image has the wrong dimensions."
        other = sub[sub.status != "PASS"].status.value_counts().to_dict()
        print(f"{s}x{s} | {ALGO_LABELS[m]:20s}: {len(ok)}/{len(sub)} exact clean recovery | other: {other or 'none'}")

assert all(p["split"] == "test" for p in EVAL_PAIR_LIST)
assert not ({p["pair_id"] for p in EVAL_PAIR_LIST} & {p["pair_id"] for p in train_pairs + val_pairs}), "Test pair found in training data."
saved_pairs = pd.read_csv(PAIR_FILE)
assert saved_pairs[["pair_id", "cover_file", "secret_file"]].astype(str).equals(pairs_df[["pair_id", "cover_file", "secret_file"]].astype(str)), "Pair assignment changed."
assert check_model_compatible(ppo_model) is None and os.path.abspath(DEPLOYED_MODEL_ZIP).startswith(_new_root + os.sep)
assert len(glob.glob(os.path.join(CHECKPOINT_DIR, f"{CKPT_PREFIX}_*_steps.zip"))) > 0, "No checkpoint was saved in the new folder."
if WARM_INFO.get("old_sha256"):
    assert sha256_file(OLD_MODEL_ZIP) == WARM_INFO["old_sha256"], "The previous model file was modified!"
assert snapshot_files(EXISTING_PATHS) == EXISTING_SNAPSHOT, "An existing project file (including the previous experiment) was changed or added."
assert snapshot_files(DATASET_DIRS) == DATASET_SNAPSHOT, "A dataset file was changed or added."
print("\nAll methods used the same test pairs. Pair assignments are unchanged. Previous experiment, models and datasets are untouched.")

print(f"\nDatasets: cover = {SELECTED_COVER_DATASET} | secret = {SELECTED_SECRET_DATASET}")
print(f"Pairs: train {len(train_pairs)} | validation {len(val_pairs)} | test {len(test_pairs)} (evaluated: {len(EVAL_PAIR_LIST)})")
print(f"PPO model: {os.path.basename(DEPLOYED_MODEL_ZIP)} ({ppo_model.num_timesteps} timesteps, SHA-256 {MODEL_SHA256[:16]}...) | "
      f"warm start: {bool(WARM_INFO.get('warm_start'))}")
print("\nClean results (mean over embedded pairs):")
print(summary_df[["Algorithm", "Secret image size", "Pairs embedded", "MSE", "PSNR (dB)", "SSIM", "BER", "Bit accuracy",
                  "Exact recovery rate (%)", "Extraction completed (%)"]].round(4).to_string(index=False))
if len(N):
    print(f"\nNoise experiments: {len(N)} trials | extraction completed {100*istrue(N.extraction_completed).mean():.2f}% | "
          f"mean BER {mean_ok(N.ber):.4f} | mean bit accuracy {mean_ok(N.bit_accuracy):.4f} | "
          f"exact recovery {100*istrue(N.exact_recovery).mean():.2f}% | mean recovered-secret SSIM {mean_ok(N.secret_ssim):.4f}")
print("\nOutput directories:")
for name, p in [("Processed data", PROCESSED_DIR), ("Models", MODEL_DIR), ("Checkpoints", CHECKPOINT_DIR), ("Logs", LOG_DIR),
                ("Stego images", STEGO_DIR), ("Metadata", META_DIR), ("Recovered images", RECOVERED_DIR), ("Tables", TABLE_DIR),
                ("Graphs", PLOT_DIR), ("Noise experiments", NOISE_DIR), ("Comparison results", COMPARISON_DIR), ("Sample outputs", SAMPLE_DIR)]:
    print(f"  {name}: {p}")
print("Final verification completed.")